# Practical 04 — Digital Elevation Models and Hydrological Processes
### Satellite Remote Sensing Practical Series

**Version 4 — ROI geometry definition and SRTM masking fix.** · PLNT 7620 (Part 4) · Digital Agroecosystems Lab, University of Manitoba
**Case study:** Elm Creek, Manitoba · **Tools:** Google Colab, Google Earth Engine (GEE), LiDAR, SRTM, Rasterio and WhiteboxTools

## Purpose and research question
**How do elevation-data source and pixel size influence our interpretation of agricultural drainage, topographic wetness, closed depressions and potential surface-water connectivity?**

Satellite remote sensing is not limited to vegetation mapping. Elevation information supplies a geometric framework for studying runoff pathways, field-scale water accumulation, erosion risk and landscape organization. In prairie terrain, elevation differences of only a few decimetres may redirect simulated flow. The same landscape can therefore appear markedly different in a 1 m LiDAR DEM and a 30 m satellite-derived DEM.

### Our three-way comparison
| DEM / product | Nominal working grid | What it helps us test |
|:--|:--|:--|
| **A — Airborne LiDAR (native)** | 1 m | Fine-scale ridges, ditches, swales and potholes |
| **C — LiDAR aggregated** | 30 m | The effect of coarsening **the same elevation source** |
| **B — SRTM** | 30 m, resampled to projected grid | Differences between **data sources** on an aligned 30 m grid |

The notebook also evaluates **SRTM over the full regional ROI**, as opposed to the smaller, LiDAR-covered comparison area. Compare A–C to study resolution effects; compare C–B to study source effects. These comparisons are useful, but they cannot completely separate differences in acquisition date, vertical reference, surface representation or resampling.

### Learning outcomes
By the end of this practical, you should be able to:
1. Load, inspect, clip, reproject and align elevation rasters for a valid comparison.
2. Explain the difference between a raw and a hydrologically conditioned DEM.
3. Calculate and interpret **eight terrain/hydrology analyses**: (1) elevation–slope–aspect, (2) D8 flow direction, (3) flow accumulation and drainage, (4) watersheds, (5) TWI, (6) depression mapping, (7) depression storage and (8) fill-and-spill connectivity.
4. Quantify changes in modelled drainage caused by changing data source or pixel size.
5. Distinguish **topographic potential** from observed water flow and evaluate uncertainty in agricultural landscapes.
6. Save reproducible maps, GIS data and statistical tables into **your own Google Drive**.

### What is expected of you
Run cells sequentially, inspect figures and diagnostic messages, and record quantitative findings instead of guessing outcomes. The analysis uses a **provided** Elm Creek ROI and LiDAR extent; the optional QUICK subset is recommended for initial testing. **Each student must use their own Earth Engine project and Drive**. Code assistance is allowed where permitted by the instructor, but the scientific interpretation must be your own.


## 0. Before you begin — student instructions

**You do not need to locate, upload, draw, or type any file paths for the class data.** This notebook obtains all of the instructor-provided boundaries and LiDAR from the two linked shared Google Drive folders automatically.

1. Open in Google Colab and select **File → Save a copy in Drive**.
2. Complete the three fields in **Student Configuration**: your Google Earth Engine **project ID**, a unique **student identifier**, and **QUICK/FULL** processing mode. There are **no file-path fields** to complete.
3. Select **Runtime → Run all** (or run cells in order), authorize Google Drive and Earth Engine using **your own account**, and review maps/figures as you proceed.
4. The notebook automatically retrieves **all five parts of `ROI_1`** (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`) from the instructor's [ROI folder](https://drive.google.com/drive/folders/1Na-g_HROPwbgsYLJ3Cj9u3xCFJmFUz-s).
5. It also retrieves **`Zoom_roi.kmz`** and the **required LiDAR ZIP(s)** from the instructor's [class Data folder](https://drive.google.com/drive/folders/1i3Ovt3cPGgTaw5rgfruavjWBoqSBFDAb). **QUICK** generally requires only `Elm_Creek_LiDAR_DEM2.zip`; FULL can require both archives.
6. SRTM is retrieved automatically through your Earth Engine account. Everything produced by the exercise is saved in **your own Google Drive**: `MyDrive/Satellite_Practicals/Practical_04/<your_identifier>/`.

**No manual downloads or paths are part of the normal workflow.** If the instructor's shared folder blocks automated access or Drive rate-limits a large file, the notebook will give a clear **data-access error** so the instructor can correct the sharing/transfer problem. An inaccessible file is not replaced with invented data.

**Research caution.** Results describe DEM-derived runoff pathways and potential depressional storage, not observed flood depths or calibrated discharge. Spatial resolution, vertical error, terrain conditioning, and watershed outlet selection affect interpretations.


In [ ]:
#@title Practical 4 — Student Configuration (Google Colab Form)
# Students: ONLY change these three fields. No file paths or uploads required.
GEE_PROJECT_ID = "CHANGE_ME"  #@param {type:"string"}
STUDENT_FOLDER = "student_001"  #@param {type:"string"}
MODE = "QUICK"  #@param ["QUICK", "FULL"]

# ====================== INSTRUCTOR CLASS DATA ==============================
# The regional shapefile and zoom/LiDAR sources are predefined for everyone.
# This is fixed course configuration, NOT a student input form.
ROI_FOLDER_ID = "1Na-g_HROPwbgsYLJ3Cj9u3xCFJmFUz-s"
DATA_FOLDER_ID = "1i3Ovt3cPGgTaw5rgfruavjWBoqSBFDAb"
ROI_FOLDER_URL = f"https://drive.google.com/drive/folders/{ROI_FOLDER_ID}"
LIDAR_FOLDER_URL = f"https://drive.google.com/drive/folders/{DATA_FOLDER_ID}"
SAVE_TO_DRIVE = True  # Required: all outputs go to the student's own Drive.

# Hydrological options preserved from the original Elm Creek exercise.
OUTLET_X, OUTLET_Y = 569450, 5508000  # metres, NAD83/UTM 14N
SNAP_DISTANCE_M = 90
STREAM_THRESHOLD_HA = 10
MIN_DEPRESSION_DEPTH_M = 0.10
MIN_DEPRESSION_AREA_M2 = 100
CONDITIONING = "fill"
EXPORT_NATIVE_RASTERS = True

print("Source data: automatically provided by the instructor's shared Drive folders.")
print("Student output folder:", STUDENT_FOLDER)
print("Processing mode:", MODE)


## 1. Study area and elevation datasets
**Elm Creek** lies on the former bed of glacial **Lake Agassiz**, west of the Red River Valley. The **LiDAR zoom area**
(`Zoom_roi.kmz`, ≈ 3.4 km × 8.3 km, drawn by your instructor) straddles an old Lake Agassiz **beach ridge / strandline** that
runs north-west to south-east. To its west is the slightly higher lake-plain surface (≈ 258–265 m) with many small **closed
depressions**; to its east is the lower, very flat plain (≈ 250–254 m). Small **gullies** cut through the ridge, and a
1.6 km **grid of roads and ditches** crosses everything. The terrain is extremely flat: most slopes are below 1°, so a few
decimetres of elevation decide where water goes.

| | **LiDAR DEM** | **SRTM DEM** |
|---|---|---|
| Source | Provincial airborne LiDAR, class folder (2 ZIP archives) | NASA Shuttle Radar Topography Mission, `USGS/SRTMGL1_003` in Earth Engine |
| Contents | `Elm_Creek_LiDAR_DEM1.zip` → one GeoTIFF (29,732 × 34,896 cells, west tile); `Elm_Creek_LiDAR_DEM2.zip` → one GeoTIFF (31,692 × 34,896 cells, east tile). The two tiles meet at x ≈ 566,735 m | Global 1 arc-second grid, band `elevation` |
| Pixel size | **1 m** | ≈ 30 m (1 arc-second); resampled here to a 30 m UTM grid |
| Coordinate system | NAD83 / UTM zone 14N (EPSG:26914), metres | WGS84 geographic → reprojected to EPSG:26914 |
| Elevation units / NoData | metres (float32) / 0 | metres (integers in the source) / – |
| Surface represented | **Bare earth** (buildings and trees removed; roads, ditches and dugouts remain) | **Mixed surface**: C-band radar partly reflects crops, trees and buildings |
| Date | File timestamps January 2016 (acquisition year is not stored in the files) | February 2000 |
| Vertical reference | Not stored in the GeoTIFF; Manitoba LiDAR is normally referenced to the CGVD28 geoid (check the provider's metadata) | EGM96 geoid |
| Coverage of the ROI | ≈ 10,700 ha of the 48,500 ha ROI (north band and western strip) | Whole ROI |

Because the LiDAR covers only part of the ROI, the three-way comparison is made in the **zoom area, which lies completely
inside the LiDAR coverage**, and SRTM gives the regional picture for the whole ROI. In `QUICK` mode only the central
3 km × 3 km of the zoom area is analysed (clearly marked on every regional map).

| Data in the class folder (instructor’s shared practical-data folder) | Content |
|---|---|
| `Elm_Creek_LiDAR_DEM1.zip`, `Elm_Creek_LiDAR_DEM2.zip` | LiDAR DEM tiles (1.8 + 1.5 GB) |
| `roi/ROI_1.*` | Study-area boundary (Shapefile) |
| `Zoom_roi.kmz` | LiDAR zoom area (Google Earth polygon) |

💡 **LiDAR is not "the truth".** It is far more detailed and accurate than SRTM (expected accuracy is often higher than SRTM, but product-specific accuracy must be checked), but it was collected 15 years later, it shows roads and ditches that may contain culverts the DEM cannot see,
and its own noise can create tiny false depressions.
### Remote sensing interpretation: DEM versus DSM
An airborne LiDAR *bare-earth DEM* attempts to represent the ground after filtering above-ground returns. Radar-derived SRTM elevations may include canopy and built-surface effects, and both products differ in acquisition period and vertical reference. A difference between these DEMs is therefore **not automatically a terrain change**. Inspect metadata and use field evidence before making that claim.

**Data provenance:** Your instructor supplies the LiDAR and ROI source files; the script obtains the public `USGS/SRTMGL1_003` image from GEE. The 1 arc-second SRTM source is resampled to a common metric grid for comparison; this does not create new terrain detail.


## 2. Environment setup — preserve the Colab environment
We import the Python packages required for raster processing, GIS, figures, network analysis, and Google Earth Engine. **Only missing libraries will be installed.** Avoid upgrading `google-auth`, `pandas`, `Pillow` or the Earth Engine API as a group: Colab runtime requirements and library requirements can conflict.

**What to check:** A successful cell reports `All required packages are available`. If installation explicitly asks for a restart, restart once and rerun the notebook from the top.


In [ ]:
import sys, subprocess, importlib.util
IN_COLAB = 'google.colab' in sys.modules
REQUIRED = {'rasterio': 'rasterio>=1.3', 'whitebox': 'whitebox==2.3.6', 'gdown': 'gdown>=5.0',
            'ee': 'earthengine-api>=1.0', 'networkx': 'networkx>=3.0', 'geopandas': 'geopandas>=0.14'}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('All required packages are available.')

## 3. Connect to your own Google Drive and GEE project
**Authentication and permissions are personal.** Mount your own Google Drive when prompted, then sign in to an Earth Engine account allowed to use the Cloud project entered above. A shared class data link does **not** mean you should use the instructor's project ID.

The notebook will create folders for the eight analyses and source-data quality checks inside **your own** Drive. It also retains intermediate DEM files so repeat runs can reuse expensive processing steps. If Drive is not mounted, the notebook must be treated as temporary Colab storage, which is lost when the runtime is deleted.


In [ ]:
import os, json, time, math, shutil, warnings, zipfile, re, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
from rasterio.features import shapes, geometry_mask
from rasterio.transform import from_origin
from shapely.geometry import Point, LineString, box, shape
from scipy import ndimage
import networkx as nx
import requests
import ee
import gdown
from IPython.display import display
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

# The class data folders are specified by the instructor; no student data paths.
# Students authenticate only to their own Google Drive and GEE project.

MODE = MODE.upper().strip()
if MODE not in ('QUICK', 'FULL'):
    raise ValueError('MODE must be "QUICK" or "FULL" (settings cell).')

# --- Each student writes to a distinct folder in THEIR OWN Google Drive. ---
STUDENT_FOLDER_SAFE = re.sub(r'[^A-Za-z0-9_-]+', '_', str(STUDENT_FOLDER)).strip('_')
if not STUDENT_FOLDER_SAFE or STUDENT_FOLDER_SAFE.upper() in ('CHANGE_ME', 'ENTER_STUDENT_ID'):
    raise ValueError('Edit STUDENT_FOLDER in the settings cell to a unique non-sensitive label.')

# --- Google Drive ---
DRIVE_ROOT = Path('/content/drive/MyDrive')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
if IN_COLAB:
    if not DRIVE_ROOT.exists():
        raise RuntimeError('Google Drive did not mount. Rerun this cell and approve access so results are saved to YOUR Drive.')
    OUT_ROOT = DRIVE_ROOT / 'Satellite_Practicals' / 'Practical_04' / STUDENT_FOLDER_SAFE
else:
    OUT_ROOT = Path(os.environ.get('L24_OUTPUT_DIR', Path.cwd() / 'Satellite_Practicals' / 'Practical_04' / STUDENT_FOLDER_SAFE))
WORK = Path('/content/l24_work') if IN_COLAB else Path(os.environ.get('L24_WORK_DIR', Path.cwd() / 'l24_work'))
CACHE = OUT_ROOT / '_cache'
OUT = {k: OUT_ROOT / v for k, v in {
    '01': '01_Elevation_Slope', '02': '02_Flow_Direction', '03': '03_Flow_Accumulation', '04': '04_Watersheds',
    '05': '05_TWI', '06': '06_Depressions', '07': '07_Storage_Capacity', '08': '08_Connectivity',
    'S': '00_Dataset_Quality'}.items()}
for p in [*OUT.values(), CACHE, WORK]:
    p.mkdir(parents=True, exist_ok=True)
print('Results folder :', OUT_ROOT)

# --- Google Earth Engine (your own Cloud project) ---
if not GEE_PROJECT_ID or GEE_PROJECT_ID in ('CHANGE_ME', 'ENTER_PROJECT_ID'):
    raise ValueError('Type your Earth Engine Cloud project ID into GEE_PROJECT_ID in the settings cell '
                     '(enter the project registered for YOUR Earth Engine account).')
try:
    if IN_COLAB:
        ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    ee.Number(1).getInfo()
except Exception as exc:
    raise RuntimeError(
        f'Earth Engine could not start with project "{GEE_PROJECT_ID}".\n'
        'Check (1) the project ID spelling, (2) that the project is registered for Earth Engine '
        '(https://console.cloud.google.com/earth-engine), and (3) that you signed in with the same Google account.\n'
        f'Original error: {exc}') from exc
print('Earth Engine   : connected with project', GEE_PROJECT_ID)

# --- WhiteboxTools ---
import whitebox
WBT = whitebox.WhiteboxTools()
WBT.set_verbose_mode(False)
print('WhiteboxTools  :', WBT.version().splitlines()[0])

### Reusable processing toolbox — what you need to understand
This longer code cell contains functions reused throughout the notebook. You **do not** need to modify them for the first run, but you should know what they do:

| Function group | Responsibility | Why it matters |
|---|---|---|
| `read_raster`, `write_raster` | Read/write georeferenced rasters with a NoData mask | Prevents missing values being interpreted as zero elevation |
| `block_mean`, `upsample` | Align comparisons between 1 m and 30 m | Spatial resolution must be handled explicitly |
| `wbt_run`, `run_hydrology` | Run WhiteboxTools, produce conditioned DEM and D8 surfaces | Hydrologic routing requires a consistent preprocessing rule |
| `compare_maps`, `regional_map`, `diff_stats` | Maps, statistics and comparable extents | Visual differences alone are not quantitative evidence |
| Depression/network helpers | Identify sinks, estimate volumes and trace potential paths | These are *topographic models*, not observations of water |

**Important:** The raw DEM is retained for depression analysis; a separate conditioned DEM is created for routing. Never interpret depression counts computed from a filled DEM as the original landscape's closed depressions.


#### Spatial grids: why metres, NoData and alignment matter
The analysis uses **EPSG:26914 (NAD83 / UTM zone 14N)** so that distances, areas and slopes can be computed in metres. The `GRID=30` setting defines a metric 30 m comparison grid. For a 30 m raster, one cell represents **900 m² = 0.09 ha**; a 1 m cell represents **1 m²**. A fixed number of upstream cells must therefore never be interpreted as the same contributing area at both resolutions.

The source LiDAR is resampled through an **area-weighted mean** to form a 30 m comparison DEM. SRTM is interpolated onto that grid. Grid alignment prevents a half-pixel shift from being mistaken for a real terrain difference. NoData pixels must remain masked.


In [ ]:
# ----------------------------- constants -----------------------------
CRS_EPSG = 26914                      # NAD83 / UTM zone 14N: the LiDAR's own projected metric CRS
CRS = f'EPSG:{CRS_EPSG}'
GRID = 30.0                           # common comparison grid (m)
AGG = 30                              # 1 m cells per 30 m cell along each axis
QUICK_BOX = (567750, 5506860, 570750, 5509860)   # 3 km x 3 km in the centre of the zoom area (EPSG:26914, on the 30 m grid)
PRODUCTS = ['LiDAR_1m', 'LiDAR_30m', 'SRTM_30m']
LABEL = {'LiDAR_1m': 'A · LiDAR 1 m (native)', 'LiDAR_30m': 'C · LiDAR aggregated to 30 m',
         'SRTM_30m': 'B · SRTM 30 m', 'SRTM_ROI': 'SRTM 30 m · whole ROI'}
SHORT = {'LiDAR_1m': 'LiDAR 1 m', 'LiDAR_30m': 'LiDAR 30 m', 'SRTM_30m': 'SRTM 30 m', 'SRTM_ROI': 'SRTM ROI'}
COLOR = {'LiDAR_1m': '#1f4e79', 'LiDAR_30m': '#4dabf7', 'SRTM_30m': '#e8590c', 'SRTM_ROI': '#e8590c'}
RES = {'LiDAR_1m': 1.0, 'LiDAR_30m': 30.0, 'SRTM_30m': 30.0, 'SRTM_ROI': 30.0}
# Typical vertical uncertainty, used to flag depressions that are shallower than the DEM's own noise.
# Illustrative thresholds only; NOT verified RMSE values for Elm Creek. Confirm survey metadata before scientific reporting.
VERTICAL_UNCERTAINTY_M = {'LiDAR_1m': 0.15, 'LiDAR_30m': 0.15, 'SRTM_30m': 2.0, 'SRTM_ROI': 2.0}
BREACH_DISTANCE_M = 100               # only used when CONDITIONING = "breach"
MAX_DISPLAY_PX = 1500                 # maps of 1 m rasters are subsampled for display only (analysis stays at 1 m)
SUMMARY = []                          # rows for the final comparison table

# ----------------------------- raster I/O -----------------------------
def read_raster(path):
    """Return (array float32 with NaN for NoData, affine transform)."""
    with rasterio.open(path) as src:
        a = src.read(1, masked=True).astype('float32').filled(np.nan)
        return a, src.transform

def write_raster(path, arr, transform, compress=True, dtype='float32', nodata=-9999.0):
    """Write a single-band GeoTIFF in EPSG:26914. compress=False gives a plain file for WhiteboxTools."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    data = np.where(np.isfinite(arr), arr, nodata).astype(dtype)
    prof = dict(driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1, dtype=dtype, crs=CRS,
                transform=transform, nodata=nodata, BIGTIFF='IF_SAFER')
    if compress:
        prof.update(compress='deflate', tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(path, 'w', **prof) as dst:
        dst.write(data, 1)
    return path

def export_raster(arr, tag, name, folder):
    """Save an analysis raster to the student's output folder (1 m rasters only if EXPORT_NATIVE_RASTERS)."""
    if tag == 'LiDAR_1m' and not EXPORT_NATIVE_RASTERS:
        return None
    return write_raster(Path(folder) / f'{name}_{tag}.tif', arr, TRANSFORM[tag])

# ----------------------------- WhiteboxTools wrapper -----------------------------
def wbt_run(tool, **kwargs):
    """Run one WhiteboxTools tool, WAIT for it to finish and raise a readable error if it fails.
    The engine is called directly (subprocess.run) so that errors are never hidden.
    Keyword names are the WhiteboxTools command-line flags, e.g. dem=..., output=..., fix_flats=True."""
    args = [str(Path(WBT.exe_path) / WBT.exe_name), '--run=' + ''.join(w.capitalize() for w in tool.split('_'))]
    for k, v in kwargs.items():
        if v is True:
            args.append(f'--{k}')
        elif v is not False and v is not None:
            args.append(f'--{k}={v}')
    if tool.startswith('fill_depressions'):
        args.append('--max_procs=1')    # the multi-threaded fill occasionally crashes (WhiteboxTools 2.4); 1 thread is stable
    args += ['-v=false', '--compress_rasters=False']
    out = kwargs.get('output')
    for attempt in range(3):            # retry: rare transient failures are harmless to repeat
        p = subprocess.run(args, cwd=WBT.exe_path, capture_output=True, text=True)
        if p.returncode == 0 and (not out or Path(out).exists()):
            return
    if True:
        raise RuntimeError(f'WhiteboxTools "{tool}" failed (code {p.returncode}).\nCommand: {" ".join(args[1:])}\n'
                           f'{(p.stdout + p.stderr)[-1500:]}')

# ----------------------------- grid helpers -----------------------------
def block_mean(a, f=AGG, min_valid=0.9):
    """Area-weighted mean of f x f blocks. Because the 1 m cells nest exactly inside the 30 m cells, a plain block
    mean IS the area-weighted mean. Blocks with < min_valid valid cells become NoData."""
    h, w = a.shape
    b = a.reshape(h // f, f, w // f, f)
    v = np.isfinite(b)
    n = v.sum(axis=(1, 3))
    s = np.where(v, b, 0).astype('float64').sum(axis=(1, 3))
    out = (s / np.maximum(n, 1)).astype('float32')
    out[n < min_valid * f * f] = np.nan
    return out

def block_std(a, f=AGG):
    h, w = a.shape
    return np.nanstd(a.reshape(h // f, f, w // f, f), axis=(1, 3)).astype('float32')

def upsample(a, f=AGG):
    """Repeat each 30 m cell as 30 x 30 one-metre cells (exact, because the grids are nested)."""
    return np.repeat(np.repeat(a, f, axis=0), f, axis=1)

def diff_stats(test, ref):
    """Bias, MAE, RMSE and Pearson r of test - ref over cells valid in both."""
    m = np.isfinite(test) & np.isfinite(ref)
    d = (test - ref)[m].astype('float64')
    return {'n_cells': int(m.sum()), 'bias_m': d.mean(), 'MAE_m': np.abs(d).mean(),
            'RMSE_m': np.sqrt((d ** 2).mean()), 'r': np.corrcoef(test[m], ref[m])[0, 1]}

def hillshade(dem, res, azimuth=315, altitude=45, zfactor=3):
    """Hillshade with FIXED illumination and no contrast stretch, so all DEMs are shaded identically."""
    z = np.where(np.isfinite(dem), dem, np.nanmean(dem)).astype('float64')
    gy, gx = np.gradient(z, res)                      # gy: change towards the south, gx: towards the east
    slope = np.arctan(zfactor * np.hypot(gx, gy))
    aspect = np.arctan2(gy, -gx)
    zen, az = np.radians(90 - altitude), np.radians(360 - azimuth + 90)
    hs = np.cos(zen) * np.cos(slope) + np.sin(zen) * np.sin(slope) * np.cos(az - aspect)
    return np.clip(hs, 0, 1).astype('float32')

def extent_of(tag):
    a = SHAPE[tag]; t = TRANSFORM[tag]
    return [t.c, t.c + t.a * a[1], t.f + t.e * a[0], t.f]

def add_summary(analysis, metric, unit, values):
    SUMMARY.append({'Analysis': analysis, 'Metric': metric, 'Unit': unit,
                    **{SHORT[p]: values.get(p, np.nan) for p in PRODUCTS}})

# ----------------------------- D8 helpers -----------------------------
# WhiteboxTools D8 pointer codes:  64 128 1 / 32 0 2 / 16 8 4   (0 = no downslope neighbour / outlet)
D8 = {1: (-1, 1, 'NE'), 2: (0, 1, 'E'), 4: (1, 1, 'SE'), 8: (1, 0, 'S'),
      16: (1, -1, 'SW'), 32: (0, -1, 'W'), 64: (-1, -1, 'NW'), 128: (-1, 0, 'N')}

def downstream_index(pntr):
    """Flat index of the D8 receiver of every cell (-1 = flows off the grid / no receiver)."""
    h, w = pntr.shape
    code = np.nan_to_num(pntr, nan=0).astype(np.int64)
    dr = np.zeros_like(code); dc = np.zeros_like(code)
    for k, (r, c, _) in D8.items():
        m = code == k; dr[m] = r; dc[m] = c
    rr = np.arange(h)[:, None] + dr
    cc = np.arange(w)[None, :] + dc
    ok = (code > 0) & (rr >= 0) & (rr < h) & (cc >= 0) & (cc < w)
    return np.where(ok, rr * w + cc, -1).ravel()

def first_downstream_label(nxt, labels):
    """For every cell, the first depression (label > 0) reached by following D8 flow (0 = none).
    Uses pointer jumping, so it needs only ~log2(path length) vectorised passes."""
    res = labels.ravel().astype(np.int64).copy()
    done = (res > 0) | (nxt < 0)
    ptr = nxt.copy()
    while True:
        und = np.flatnonzero(~done)
        if und.size == 0:
            return res
        j = ptr[und]
        dj = done[j]
        res[und[dj]] = res[j[dj]]
        done[und[dj]] = True
        nd = und[~dj]
        ptr[nd] = ptr[ptr[nd]]

# ----------------------------- hydrological pipeline -----------------------------
def run_hydrology(tag, dem_path):
    """Standard WhiteboxTools workflow, identical for every DEM. Two DEM versions are kept apart:
       * ORIGINAL (unfilled) DEM -> slope, aspect, depressions and storage (natural depressions preserved)
       * CONDITIONED DEM         -> flow direction, flow accumulation, streams, watersheds, TWI (continuous routing)"""
    res = RES[tag]
    w = WORK / MODE / tag; w.mkdir(parents=True, exist_ok=True)
    f = {k: str(w / f'{k}.tif') for k in ['filled', 'cond', 'pntr', 'acc_cells', 'sca', 'slope', 'aspect',
                                          'streams', 'subbasins']}
    f['streams_shp'] = str(w / 'streams.shp'); f['dem'] = str(dem_path)
    key = json.dumps({'cond': CONDITIONING, 'thr': STREAM_THRESHOLD_HA, 'breach': BREACH_DISTANCE_M,
                      'dem_mtime': os.path.getmtime(dem_path)}, sort_keys=True)
    if (w / 'settings.json').exists() and (w / 'settings.json').read_text() == key and \
            all(Path(v).exists() for v in f.values()):
        print(f'  {SHORT[tag]:10s}: reusing cached results'); return f
    t0 = time.time()
    # depression-filled surface WITHOUT flat gradients: used only to measure depression depth / spill level
    wbt_run('fill_depressions', dem=f['dem'], output=f['filled'], fix_flats=False)
    # conditioned DEM for continuous flow routing
    if CONDITIONING == 'fill':
        wbt_run('fill_depressions', dem=f['dem'], output=f['cond'], fix_flats=True)
    elif CONDITIONING == 'breach':
        wbt_run('breach_depressions_least_cost', dem=f['dem'], output=f['cond'],
                dist=max(1, int(round(BREACH_DISTANCE_M / res))), fill=True)
    else:
        raise ValueError('CONDITIONING must be "fill" or "breach"')
    wbt_run('d8_pointer', dem=f['cond'], output=f['pntr'])
    wbt_run('d8_flow_accumulation', input=f['pntr'], output=f['acc_cells'], out_type='cells', pntr=True)
    wbt_run('d8_flow_accumulation', input=f['pntr'], output=f['sca'], out_type='specific contributing area', pntr=True)
    wbt_run('slope', dem=f['dem'], output=f['slope'], units='degrees')
    wbt_run('aspect', dem=f['dem'], output=f['aspect'])
    thr_cells = STREAM_THRESHOLD_HA * 1e4 / res ** 2
    wbt_run('extract_streams', flow_accum=f['acc_cells'], output=f['streams'], threshold=thr_cells,
            zero_background=True)
    wbt_run('raster_streams_to_vector', streams=f['streams'], d8_pntr=f['pntr'], output=f['streams_shp'])
    wbt_run('subbasins', d8_pntr=f['pntr'], streams=f['streams'], output=f['subbasins'])
    (w / 'settings.json').write_text(key)
    print(f'  {SHORT[tag]:10s}: done in {time.time() - t0:.0f} s')
    return f

_LOADED = {}
def R(tag, name):
    """Load (and cache) one hydrological raster for one DEM product, masked to the common footprint."""
    k = (tag, name)
    if k not in _LOADED:
        a, _ = read_raster(HY[tag][name])
        if name != 'dem':
            a[~np.isfinite(R(tag, 'dem'))] = np.nan
        _LOADED[k] = a
    return _LOADED[k]

# ----------------------------- depressions -----------------------------
def find_depressions(tag, min_depth=None, min_area=None):
    """Closed depressions from the ORIGINAL DEM: depth = (depression-filled surface) - (original DEM).
    Returns the label raster, the depth raster and an inventory table."""
    min_depth = MIN_DEPRESSION_DEPTH_M if min_depth is None else min_depth
    min_area = MIN_DEPRESSION_AREA_M2 if min_area is None else min_area
    res = RES[tag]; ca = res * res
    dem = R(tag, 'dem'); filled = R(tag, 'filled')
    depth = np.where(np.isfinite(dem) & np.isfinite(filled), filled - dem, 0).astype('float32')
    depth[depth < 1e-4] = 0
    lab, n = ndimage.label(depth > 0, structure=np.ones((3, 3)))
    cols = ['dep_id', 'n_cells', 'area_m2', 'max_depth_m', 'mean_depth_m', 'volume_m3', 'spill_elev_m',
            'bottom_elev_m', 'x', 'y', 'touches_edge', 'class']
    if n == 0:
        return np.zeros(dem.shape, np.int32), np.zeros_like(depth), pd.DataFrame(columns=cols)
    ids = np.arange(1, n + 1)
    cells = np.bincount(lab.ravel(), minlength=n + 1)[1:]
    maxd = ndimage.maximum(depth, lab, ids)
    keep = (maxd >= min_depth) & (cells * ca >= min_area)
    new = np.zeros(n + 1, np.int32); new[ids[keep]] = np.arange(1, keep.sum() + 1)
    lab = new[lab]; n = int(keep.sum())
    if n == 0:
        return lab, np.zeros_like(depth), pd.DataFrame(columns=cols)
    ids = np.arange(1, n + 1)
    depth = np.where(lab > 0, depth, 0).astype('float32')
    cells = np.bincount(lab.ravel(), minlength=n + 1)[1:]
    vol = np.bincount(lab.ravel(), weights=depth.ravel(), minlength=n + 1)[1:] * ca
    rc = np.array(ndimage.center_of_mass(lab > 0, lab, ids)).reshape(-1, 2)
    t = TRANSFORM[tag]
    frame = np.zeros(dem.shape, bool); frame[[0, -1], :] = True; frame[:, [0, -1]] = True
    border = frame | ndimage.binary_dilation(~np.isfinite(dem), structure=np.ones((3, 3)))
    edge = np.isin(ids, np.unique(lab[border & (lab > 0)]))
    df = pd.DataFrame({'dep_id': ids, 'n_cells': cells, 'area_m2': cells * ca,
                       'max_depth_m': ndimage.maximum(depth, lab, ids),
                       'mean_depth_m': vol / (cells * ca), 'volume_m3': vol,
                       'spill_elev_m': ndimage.maximum(filled, lab, ids),
                       'bottom_elev_m': ndimage.minimum(dem, lab, ids),
                       'x': t.c + (rc[:, 1] + 0.5) * t.a, 'y': t.f + (rc[:, 0] + 0.5) * t.e,
                       'touches_edge': edge})
    sigma = VERTICAL_UNCERTAINTY_M[tag]
    cls = np.full(n, 'likely natural', dtype=object)
    cls[(df.max_depth_m >= 1.5) & (df.area_m2 <= 3000) & (res <= 2)] = 'possibly human-made (dugout/pit)'
    cls[(df.max_depth_m < sigma) | (df.n_cells < 3)] = 'uncertain (within DEM noise)'
    cls[edge] = 'edge (spill point may lie outside area)'
    df['class'] = cls
    return lab, depth, df[cols]

def depression_polygons(lab, df, tag):
    if len(df) == 0:
        return gpd.GeoDataFrame(df, geometry=[], crs=CRS)
    geoms = {}
    for g, v in shapes(lab.astype('int32'), mask=lab > 0, transform=TRANSFORM[tag]):
        geoms.setdefault(int(v), []).append(shape(g))
    from shapely.ops import unary_union
    gdf = gpd.GeoDataFrame(df.copy(), geometry=[unary_union(geoms[i]) for i in df.dep_id], crs=CRS)
    gdf['product'] = SHORT[tag]
    return gdf

# ----------------------------- D8 directional arrows -----------------------------
def overlay_d8_arrows(ax, pointer, transform, spacing_m=240, arrow_fraction=0.58,
                      color='#111111', max_arrows=240):
    """Draw sampled compass arrows using Whitebox D8 pointer codes.

    Each arrow is located at the centre of a valid DEM cell; samples are chosen on
    a uniform map-distance grid to avoid obscuring the D8 colour raster. Importantly,
    directions use +Easting to the right and +Northing upward, matching the map.
    This is a *display* subsample only; it does not change any hydrological result.
    """
    height, width = pointer.shape
    cell_size = abs(float(transform.a))
    # Allow enough spacing for readability on the three side-by-side panels.
    stride = max(1, int(round(spacing_m / cell_size)))
    rows = np.arange(stride // 2, height, stride, dtype=int)
    cols = np.arange(stride // 2, width, stride, dtype=int)
    if rows.size * cols.size > max_arrows:
        stride = max(stride, int(math.ceil(max(height, width) / math.sqrt(max_arrows))))
        rows = np.arange(stride // 2, height, stride, dtype=int)
        cols = np.arange(stride // 2, width, stride, dtype=int)
    rr, cc = np.meshgrid(rows, cols, indexing='ij')
    codes = pointer[rr, cc]
    valid = np.isfinite(codes) & np.isin(codes, list(D8.keys()))
    if not valid.any():
        return 0
    rr, cc, codes = rr[valid], cc[valid], codes[valid].astype(int)
    x = transform.c + (cc + 0.5) * transform.a
    y = transform.f + (rr + 0.5) * transform.e
    # The Whitebox tuple stores row and column offsets (row+ means south).
    dr = np.array([D8[int(code)][0] for code in codes])
    dc = np.array([D8[int(code)][1] for code in codes])
    length = stride * cell_size * arrow_fraction
    u = dc * length / np.hypot(dc, dr)
    v = -dr * length / np.hypot(dc, dr)
    ax.quiver(x, y, u, v, angles='xy', scale_units='xy', scale=1,
              pivot='mid', color=color, width=0.0040, headwidth=4.0,
              headlength=5.5, headaxislength=4.5, zorder=8)
    return len(codes)

# ----------------------------- plotting -----------------------------
def _disp(a):
    s = max(1, int(math.ceil(max(a.shape) / MAX_DISPLAY_PX)))
    return a[::s, ::s], s

def compare_maps(arrays, title, cbar_label, cmap='viridis', vmin=None, vmax=None, norm=None, fname=None,
                 folder=None, hill=True, overlay=None, legend=None, alpha=0.85, extend='neither'):
    """Side-by-side maps of the three products on the SAME colour scale and extent."""
    tags = list(arrays)
    fig, axes = plt.subplots(1, len(tags), figsize=(5.4 * len(tags), 5.6), constrained_layout=True)
    axes = np.atleast_1d(axes)
    sub_note = False
    for ax, t in zip(axes, tags):
        a, s = _disp(arrays[t]); sub_note |= s > 1
        ext = extent_of(t)
        if hill:
            ax.imshow(_disp(HS[t])[0], cmap='gray', extent=ext, vmin=0, vmax=1)
        im = ax.imshow(np.ma.masked_invalid(a), cmap=cmap, vmin=None if norm else vmin, vmax=None if norm else vmax,
                       norm=norm, extent=ext, alpha=alpha if hill else 1, interpolation='nearest')
        if overlay:
            overlay(ax, t)
        ax.set_title(LABEL[t], fontsize=11, color=COLOR[t], fontweight='bold')
        ax.set_xlabel('Easting (m, UTM 14N)', fontsize=8); ax.tick_params(labelsize=7)
        ax.ticklabel_format(style='plain', useOffset=False)
    axes[0].set_ylabel('Northing (m)', fontsize=8)
    if legend:
        fig.legend(handles=legend, loc='lower center', ncol=min(8, len(legend)), fontsize=8, bbox_to_anchor=(0.5, -0.06))
    else:
        cb = fig.colorbar(im, ax=list(axes), shrink=0.8, pad=0.01, extend=extend); cb.set_label(cbar_label)
    fig.suptitle(title + ('\n(1 m map subsampled for display only; all calculations use every 1 m cell)' if sub_note else ''),
                 fontsize=12)
    if fname:
        fig.savefig(Path(folder) / fname, dpi=300, bbox_inches='tight')
    plt.show()

def regional_map(arr, title, cbar_label, cmap='viridis', vmin=None, vmax=None, norm=None, fname=None, folder=None,
                 overlay=None, hill=True, legend=None):
    """Whole-ROI SRTM map with the ROI outline and the LiDAR zoom area."""
    fig, ax = plt.subplots(figsize=(5.2, 8.2), constrained_layout=True)
    ext = extent_of('SRTM_ROI')
    if hill:
        ax.imshow(HS['SRTM_ROI'], cmap='gray', extent=ext, vmin=0, vmax=1)
    im = ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, vmin=None if norm else vmin, vmax=None if norm else vmax,
                   norm=norm, extent=ext, alpha=0.85 if hill else 1, interpolation='nearest')
    if overlay:
        overlay(ax)
    ROI_UTM.boundary.plot(ax=ax, color='k', lw=1)
    gpd.GeoSeries([ZOOM_POLY], crs=CRS).boundary.plot(ax=ax, color='red', lw=1.0, ls='--')
    gpd.GeoSeries([ANALYSIS_POLY], crs=CRS).boundary.plot(ax=ax, color='red', lw=1.8)
    zb = ZOOM_POLY.bounds
    ax.text(zb[0], zb[3] + 300, f'LiDAR zoom area (solid = analysed in {MODE})', color='red', fontsize=7)
    ax.set_title(title, fontsize=10); ax.tick_params(labelsize=7); ax.ticklabel_format(style='plain', useOffset=False)
    if legend:
        ax.legend(handles=legend, fontsize=7, loc='lower left')
    else:
        cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.01); cb.set_label(cbar_label)
    if fname:
        fig.savefig(Path(folder) / fname, dpi=300, bbox_inches='tight')
    plt.show()

def sample_values(a, n=400_000, seed=0):
    v = a[np.isfinite(a)]
    if v.size > n:
        v = np.random.default_rng(seed).choice(v, n, replace=False)
    return v

def show_table(df, fname=None, folder=None, floatfmt=3):
    if fname:
        df.to_csv(Path(folder) / fname, index=True)
    display(df.round(floatfmt))

print('Toolbox ready.')

## 4. Automatically load the two official study-area boundaries

The two class boundaries are **provided by the instructor and retrieved by the notebook**, not created by students.

- **`ROI_1.shp`** defines the regional Elm Creek area for regional-scale SRTM analysis. A shapefile has several linked files. The notebook automatically downloads `ROI_1.shp`, `.shx`, `.dbf`, `.prj`, and `.cpg` together from the **instructor ROI folder**.
- **`Zoom_roi.kmz`** defines the detailed LiDAR–SRTM comparison region. It is downloaded from the **instructor Data folder**, together with the appropriate LiDAR archive later in Section 6.

The code checks the downloaded files, validates the coordinate system, transforms boundaries to **EPSG:26914**, and selects the QUICK or FULL analysis region. It saves source boundaries to the student's own Google Drive and never recursively searches *My Drive*.


In [ ]:
# ===== Section 4: Download class boundaries automatically =====
# Data sources are defined in Student Configuration. No file-path questions.
# A: ROI_1 shapefile (5 small components) from the instructor ROI folder.
# B: Zoom_roi.kmz from the instructor's main Data folder.
import xml.etree.ElementTree as ET

ROI_CACHE = WORK / 'shared_roi_inputs'
ROI_CACHE.mkdir(parents=True, exist_ok=True)
SHARED_FOLDER_LISTINGS = {}  # reuse metadata for subsequent LiDAR downloads


def shared_file_index(folder_id):
    """List only the specified instructor folder, returning filename -> Drive ID.

    `skip_download=True` means listing the folder cannot accidentally transfer
    the 1.5–1.8 GB LiDAR archives. gdown uses the Drive folder's shared access.
    """
    if folder_id not in SHARED_FOLDER_LISTINGS:
        try:
            entries = gdown.download_folder(
                id=folder_id, output=str(WORK / 'class_folder_metadata'),
                skip_download=True, quiet=True, use_cookies=False
            )
            if not entries:
                raise RuntimeError('Folder listing returned no files.')
            index = {}
            for entry in entries:
                name = Path(entry.path).name
                if name in index:
                    raise RuntimeError(f'Duplicate filename in class folder: {name}')
                index[name] = entry.id
            SHARED_FOLDER_LISTINGS[folder_id] = index
        except Exception as exc:
            raise RuntimeError(
                'The class data folder could not be listed automatically: '
                f'https://drive.google.com/drive/folders/{folder_id}. '
                'Instructor action: verify that all students have Viewer access '
                'and downloads are permitted. No student file paths are needed. '
                f'Original error: {exc}'
            ) from exc
    return SHARED_FOLDER_LISTINGS[folder_id]


def download_class_file(file_id, target):
    """Download one specified file into Colab, with a basic completeness check."""
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        return target
    if target.exists():
        target.unlink()
    result = gdown.download(id=file_id, output=str(target), quiet=True, use_cookies=False)
    if result is None or not target.exists() or target.stat().st_size == 0:
        target.unlink(missing_ok=True)
        raise RuntimeError(f'Could not download class file {target.name}. Ask the instructor to check shared access.')
    return target


# A. Regional ROI: every shapefile component is downloaded together.
roi_shp = ROI_CACHE / 'ROI_1.shp'
roi_components = ['.shp', '.shx', '.dbf', '.prj', '.cpg']
missing = [suffix for suffix in roi_components if
           not roi_shp.with_suffix(suffix).is_file() or
           roi_shp.with_suffix(suffix).stat().st_size == 0]
if missing:
    print('Downloading class ROI_1 shapefile (including all sidecar files)...')
    roi_listing = shared_file_index(ROI_FOLDER_ID)
    for suffix in missing:
        name = 'ROI_1' + suffix
        if name not in roi_listing:
            if suffix == '.cpg':
                print('Optional .cpg was not shared; using the encoding detected by GeoPandas.')
                continue
            raise FileNotFoundError(
                f'The instructor ROI folder is missing {name}. '
                'It must contain .shp, .shx, .dbf and .prj.')
        download_class_file(roi_listing[name], roi_shp.with_suffix(suffix))

required = [roi_shp.with_suffix(ext) for ext in ('.shp', '.shx', '.dbf', '.prj')]
if any(not p.is_file() or p.stat().st_size == 0 for p in required):
    raise RuntimeError('The class ROI shapefile is incomplete. Ask the instructor to check the source folder.')
roi_gdf = gpd.read_file(roi_shp)
if roi_gdf.empty or roi_gdf.crs is None:
    raise ValueError('The shared ROI_1 shapefile is empty or missing its coordinate system.')
ROI_UTM = roi_gdf.to_crs(CRS)
ROI_UTM = ROI_UTM.loc[ROI_UTM.geometry.notna() & ~ROI_UTM.geometry.is_empty].copy()
if ROI_UTM.empty:
    raise ValueError('No valid polygon found in the supplied ROI_1 shapefile.')

# Create ONE projected polygon/multipolygon representing the full regional ROI.
# It is required later to mask the whole-region SRTM DEM (Section 7).
# union_all() combines shapefile features and preserves disjoint areas.
ROI_GEOM = (ROI_UTM.geometry.union_all()
            if hasattr(ROI_UTM.geometry, 'union_all')
            else ROI_UTM.geometry.unary_union)
if not ROI_GEOM.is_valid:
    ROI_GEOM = ROI_GEOM.buffer(0)  # repair minor topology problems
if ROI_GEOM.is_empty or ROI_GEOM.geom_type not in ('Polygon', 'MultiPolygon'):
    raise ValueError('The regional ROI must contain at least one valid polygon.')

roi_ll = ROI_UTM.to_crs('EPSG:4326')

# B. Zoom area: downloaded automatically from the instructor's class Data folder.
kmz_path = ROI_CACHE / 'Zoom_roi.kmz'
if not kmz_path.is_file() or not zipfile.is_zipfile(kmz_path):
    kmz_path.unlink(missing_ok=True)
    print('Downloading class Zoom_roi.kmz...')
    data_listing = shared_file_index(DATA_FOLDER_ID)
    if 'Zoom_roi.kmz' not in data_listing:
        raise FileNotFoundError('The instructor Data folder does not contain Zoom_roi.kmz.')
    download_class_file(data_listing['Zoom_roi.kmz'], kmz_path)
if not zipfile.is_zipfile(kmz_path):
    kmz_path.unlink(missing_ok=True)
    raise ValueError('The downloaded Zoom_roi.kmz is invalid. Ask the instructor to check the source file.')

def polygon_from_kmz(path):
    """Extract exterior polygon ring(s) from KML contained in a KMZ."""
    with zipfile.ZipFile(path) as archive:
        kml_names = [n for n in archive.namelist() if n.lower().endswith('.kml')]
        if not kml_names:
            raise ValueError('No KML file found inside Zoom_roi.kmz')
        root = ET.fromstring(archive.read(kml_names[0]))
    rings = []
    for element in root.findall('.//{*}Polygon/{*}outerBoundaryIs/{*}LinearRing/{*}coordinates'):
        coords = []
        for entry in (element.text or '').strip().split():
            xyz = entry.split(',')
            if len(xyz) >= 2:
                coords.append((float(xyz[0]), float(xyz[1])))
        if len(coords) >= 4:
            rings.append(coords)
    if not rings:
        raise ValueError('No polygon exterior rings found in Zoom_roi.kmz')
    from shapely.geometry import Polygon
    from shapely.ops import unary_union
    poly = unary_union([Polygon(r) for r in rings])
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_empty:
        raise ValueError('Invalid/empty LiDAR comparison polygon.')
    return gpd.GeoSeries([poly], crs='EPSG:4326').to_crs(CRS).iloc[0]

ZOOM_POLY = polygon_from_kmz(kmz_path)
if MODE == 'QUICK':
    ANALYSIS_POLY = box(*QUICK_BOX).intersection(ZOOM_POLY)
    if ANALYSIS_POLY.is_empty:
        raise ValueError('QUICK_BOX does not intersect Zoom_roi.kmz. Check source boundaries.')
    selection_bounds = QUICK_BOX
else:
    ANALYSIS_POLY = ZOOM_POLY
    selection_bounds = ZOOM_POLY.bounds

# Rectangular processing window follows a 30 m grid; true polygons are masked later.
ZOOM_BOUNDS = (
    math.floor(selection_bounds[0] / GRID) * GRID,
    math.floor(selection_bounds[1] / GRID) * GRID,
    math.ceil(selection_bounds[2] / GRID) * GRID,
    math.ceil(selection_bounds[3] / GRID) * GRID,
)
c = roi_ll.geometry.union_all().centroid if hasattr(roi_ll.geometry, 'union_all') else roi_ll.unary_union.centroid
print('Regional ROI      :', roi_shp)
print('LiDAR zoom boundary:', kmz_path)
print('Mode and window   :', MODE, ZOOM_BOUNDS)
print('Analysis polygon  :', round(ANALYSIS_POLY.area / 1e4, 1), 'ha')
print('Coordinate system :', ROI_UTM.crs)
print('Regional ROI mask : ready (', ROI_GEOM.geom_type, ', area', round(ROI_GEOM.area / 1e4, 1), 'ha )')

# Save student-specific source geometry and record provenance.
if SAVE_TO_DRIVE:
    roi_ll.to_file(OUT['S'] / 'regional_roi.geojson', driver='GeoJSON')
    gpd.GeoSeries([ANALYSIS_POLY], crs=CRS).to_crs('EPSG:4326').to_file(
        OUT['S'] / 'analysis_polygon.geojson', driver='GeoJSON')


### Examine the two boundaries before running terrain analysis
The next map overlays the **regional ROI**, **FULL LiDAR polygon**, and **QUICK subset** on a coloured SRTM image if the Earth Engine preview is available. This preview is for **visual orientation only**; the scientific analyses run on the projected raster grids produced later.

**Check:** Are the quick comparison box and LiDAR polygon inside the regional ROI? Do the data cover the flat agricultural landscape you expected? A displayed background image is **not** a measure of accuracy.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | REGIONAL ROI PREVIEW (FOLIUM + OPTIONAL EARTH ENGINE BACKGROUND)
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# ------------------------------------------------------------
# Section 5: Visualize Elm Creek ROI with SRTM terrain background
# ------------------------------------------------------------

import folium
import requests
import base64

# Geographic boundaries
west, south, east, north = roi_ll.total_bounds

# Create interactive map without external basemap services
m = folium.Map(
    location=[c.y, c.x],
    tiles=None,
    zoom_control=True,
    control_scale=True,
    prefer_canvas=True
)

# Automatically fit the map to the complete ROI
m.fit_bounds([[south, west], [north, east]])

# ------------------------------------------------------------
# Load SRTM elevation from Google Earth Engine
# ------------------------------------------------------------

region = ee.Geometry.Rectangle(
    [west, south, east, north],
    geodesic=False
)

srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")

try:
    # Calculate elevation visualization range
    stats = srtm.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=90,
        maxPixels=1e8
    ).getInfo()

    vmin = stats["elevation_p2"]
    vmax = stats["elevation_p98"]

    # Create coloured SRTM terrain image
    terrain = srtm.visualize(
        min=vmin,
        max=vmax,
        palette=[
            "#24476b",
            "#78a6a3",
            "#bfd4aa",
            "#e5cb88",
            "#a87543"
        ]
    )

    # Generate image from Earth Engine
    url = terrain.getThumbURL({
        "region": region,
        "dimensions": 1200,
        "format": "png"
    })

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    image_data = (
        "data:image/png;base64,"
        + base64.b64encode(response.content).decode("utf-8")
    )

    # Add SRTM terrain as background
    folium.raster_layers.ImageOverlay(
        image=image_data,
        bounds=[[south, west], [north, east]],
        name="SRTM Elevation",
        opacity=1
    ).add_to(m)

    print("SRTM terrain background loaded successfully.")

except Exception as e:
    print("SRTM terrain background unavailable:", e)
    print("Displaying ROI boundaries without terrain background.")

# ------------------------------------------------------------
# Overlay study-area boundaries
# ------------------------------------------------------------

# Whole ROI - Blue
folium.GeoJson(
    roi_ll,
    name="Whole ROI (SRTM)",
    style_function=lambda f: {
        "color": "#22577a",
        "weight": 3,
        "fillOpacity": 0.05
    }
).add_to(m)

# LiDAR FULL area - Red
folium.GeoJson(
    gpd.GeoSeries([ZOOM_POLY], crs=CRS)
       .to_crs(4326).__geo_interface__,
    name="LiDAR zoom (FULL)",
    style_function=lambda f: {
        "color": "#d62828",
        "weight": 3,
        "fillOpacity": 0.1
    }
).add_to(m)

# QUICK analysis area - Orange
folium.GeoJson(
    gpd.GeoSeries([box(*QUICK_BOX)], crs=CRS)
       .to_crs(4326).__geo_interface__,
    name="QUICK subset",
    style_function=lambda f: {
        "color": "#ef8f00",
        "weight": 3,
        "fillOpacity": 0.1
    }
).add_to(m)

# Layer controls
folium.LayerControl().add_to(m)

# Display map
display(m)

## 6. Download only the LiDAR needed for the selected area and retrieve SRTM

The instructor's [class Data folder](https://drive.google.com/drive/folders/1i3Ovt3cPGgTaw5rgfruavjWBoqSBFDAb) contains two large airborne LiDAR archives:

- `Elm_Creek_LiDAR_DEM1.zip` (about 1.82 GB)
- `Elm_Creek_LiDAR_DEM2.zip` (about 1.47 GB)

Students **do not enter a LiDAR path, upload ZIPs, or search Google Drive**. The notebook determines which archive intersects the selected QUICK/FULL window, downloads only that archive into Colab's temporary working storage, and checks that it contains a GeoTIFF. In QUICK mode, the current study area generally intersects **DEM2 only**. SRTM comes from the student's own Earth Engine project.

**Important:** The ZIP download can be large and is subject to Google Drive file sharing, bandwidth, and Colab free-space limits. Students should run QUICK first. A restricted file download is a class-source permission issue, not a problem they must fix by hunting for files.


#### How automatic data retrieval works

1. The earlier cell retrieved `ROI_1` and `Zoom_roi.kmz`; you should see their local paths and the analysis footprint reported.
2. This cell checks the existing Colab download cache, then identifies which official LiDAR archive(s) overlap the analysis window.
3. It obtains archive IDs from the same instructor Data folder and downloads only missing archives. **No student path input is used.**
4. The subsequent cells inspect the LiDAR GeoTIFFs inside the ZIPs, extract the required area, and compare them with GEE SRTM on a common 30 m grid.

**Drive storage policy:** Large source ZIP files stay in the temporary Colab working directory rather than being copied unnecessarily into every student's Drive. The resulting maps, statistics, DEM extracts and output manifest are saved to the student's personal Google Drive.


In [ ]:
# ===== Section 6.1: Locate and validate LiDAR archives, with no whole-Drive search =====
LIDAR_NAMES = ['Elm_Creek_LiDAR_DEM1.zip', 'Elm_Creek_LiDAR_DEM2.zip']
# Approximate LiDAR tile extents measured from class archives (EPSG:26914).
# These guide archive selection; true coverage will be checked from the raster headers.
ARCHIVE_EXTENT = {
    'Elm_Creek_LiDAR_DEM1.zip': (537004, 5497463, 566736, 5532359),
    'Elm_Creek_LiDAR_DEM2.zip': (566735, 5497463, 598427, 5532359),
}


def needed_archives(bounds):
    """Return only the LiDAR tiles intersecting the 30 m aligned ROI window."""
    window = box(*bounds)
    need = [name for name, tile_extent in ARCHIVE_EXTENT.items()
            if window.intersects(box(*tile_extent))]
    if not need:
        raise ValueError('Selected processing window does not overlap the LiDAR archive footprints.')
    return need


def valid_lidar_zip(path):
    """Reject incomplete ZIPs and unexpected HTML/error pages."""
    path = Path(path)
    if not path.is_file() or path.stat().st_size < 1_000_000:
        return False
    try:
        if not zipfile.is_zipfile(path):
            return False
        with zipfile.ZipFile(path) as archive:
            return any(n.lower().endswith(('.tif', '.tiff')) for n in archive.namelist())
    except (OSError, zipfile.BadZipFile):
        return False


def locate_lidar():
    needed = needed_archives(ZOOM_BOUNDS)
    found = {}
    local = WORK / 'lidar_download'
    local.mkdir(parents=True, exist_ok=True)

    # Reuse downloaded source archives from this Colab runtime, when available.
    for name in needed:
        candidate = local / name
        if valid_lidar_zip(candidate):
            found[name] = candidate

    missing = [name for name in needed if name not in found]
    if missing:
        print('Required LiDAR archives :', ', '.join(needed))
        print('Downloading only missing archive(s) from the instructor Data folder:', missing)
        print('Large downloads need adequate Colab disk space; they are saved to temporary working storage.')
        try:
            listing = shared_file_index(DATA_FOLDER_ID)  # metadata only, no ZIP transfers yet
            for name in missing:
                if name not in listing:
                    raise FileNotFoundError(f'{name} was not found in the shared Data folder.')
                destination = local / name
                # gdown resume=True can recover an earlier interrupted transfer.
                result = gdown.download(
                    id=listing[name], output=str(destination),
                    quiet=False, use_cookies=False, resume=True
                )
                if result is None or not valid_lidar_zip(destination):
                    raise RuntimeError(f'Download for {name} was incomplete or invalid.')
                found[name] = destination
        except Exception as exc:
            raise RuntimeError(
                'The required LiDAR ZIP could not be downloaded automatically from the class Data folder. '
                'Instructor action: confirm Viewer/download permissions, download quotas, and file availability. '
                'Students do not need to enter file paths or upload any archives. '
                f'Source folder: {LIDAR_FOLDER_URL}. Details: {exc}'
            ) from exc

    for name in needed:
        if name not in found or not valid_lidar_zip(found[name]):
            raise FileNotFoundError(f'Required archive not ready: {name}')
    return found

LIDAR_ZIPS = locate_lidar()
for name, path in LIDAR_ZIPS.items():
    print(f'✓ {name}: {path} ({path.stat().st_size / 1e9:.2f} GB)')


### 6.2 Inspect the archives (contents, CRS, pixel size, units, NoData)
Never process a dataset you have not inspected. The following cell checks only the archives selected for this ROI. This cell lists what is inside each ZIP and reads only the GeoTIFF header.

In [ ]:
# ===========================================================================
# PRACTICAL 04 | VERIFY PROJECTION, PIXEL SIZE AND NODATA
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

def inspect_archive(zpath):
    with zipfile.ZipFile(zpath) as z:
        members = [(i.filename, i.file_size, i.compress_size) for i in z.infolist()]
    tifs = [m[0] for m in members if m[0].lower().endswith(('.tif', '.tiff'))]
    rows = []
    for t in tifs:
        with rasterio.open(f'/vsizip/{zpath}/{t}') as src:
            rows.append({'archive': Path(zpath).name, 'raster': t, 'columns': src.width, 'rows': src.height,
                         'pixel_m': src.res[0], 'dtype': src.dtypes[0], 'nodata': src.nodata,
                         'crs': src.crs.to_string() if src.crs else 'MISSING',
                         'linear_units': src.crs.linear_units if src.crs else '?',
                         'xmin': src.bounds.left, 'ymin': src.bounds.bottom, 'xmax': src.bounds.right,
                         'ymax': src.bounds.top})
    return members, rows

ARCHIVE_INFO = []
for n, p in LIDAR_ZIPS.items():
    members, rows = inspect_archive(p)
    print(f'{n}:', ', '.join(f'{m[0]} ({m[1] / 1e9:.2f} GB)' for m in members))
    ARCHIVE_INFO += rows
archive_df = pd.DataFrame(ARCHIVE_INFO)
display(archive_df)
for r in ARCHIVE_INFO:
    if r['crs'] != CRS:
        raise ValueError(f"{r['raster']} has CRS {r['crs']}; expected {CRS}.")
    if r['linear_units'].lower() not in ('metre', 'meter', 'm'):
        raise ValueError(f"{r['raster']} horizontal units are {r['linear_units']}, not metres.")
print('✓ LiDAR archive headers verified. Inspect the reported NoData and numeric data types above.')

### 6.3 Extract the LiDAR zoom area at native 1 m resolution (DEM A)
The zoom area is cut from the archive(s) that overlap it. If it crosses the boundary between the two tiles, the tiles are
**mosaicked** automatically. The result is saved in `_cache` in your output folder, so later runs skip this step.

In [ ]:
# ===========================================================================
# PRACTICAL 04 | READ AND CLIP NATIVE LiDAR FROM ARCHIVES
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

def extract_lidar(bounds, out_path):
    if out_path.exists():
        print('Reusing cached LiDAR extract:', out_path.name); return out_path
    srcs = []
    for r in ARCHIVE_INFO:
        if box(r['xmin'], r['ymin'], r['xmax'], r['ymax']).intersects(box(*bounds)):
            srcs.append(rasterio.open(f"/vsizip/{LIDAR_ZIPS[r['archive']]}/{r['raster']}"))
    if not srcs:
        raise ValueError('The zoom area does not overlap the LiDAR tiles.')
    print(f'Reading {len(srcs)} tile(s) for {bounds} at 1 m from inside the ZIP (1-4 min the first time) ...')
    t0 = time.time()
    arr, tr = merge(srcs, bounds=bounds, res=1.0, nodata=0, dtype='float32')
    for s in srcs:
        s.close()
    a = arr[0]
    a[(a <= 0) | (a > 1000)] = np.nan          # NoData (0) and impossible values
    valid = np.isfinite(a).mean()
    print(f'  read in {time.time() - t0:.0f} s | {a.shape[1]} x {a.shape[0]} cells | valid {valid:.2%}')
    if valid < 0.99:
        raise ValueError(f'Only {valid:.1%} of the zoom area has LiDAR data. Choose a zoom area inside the coverage.')
    write_raster(out_path, a, tr)
    return out_path

LIDAR_1M_PATH = extract_lidar(ZOOM_BOUNDS, CACHE / f'LiDAR_1m_{MODE}.tif')

### 6.4 Retrieve SRTM for the whole ROI from Google Earth Engine (DEM B)
SRTM is requested directly in **EPSG:26914 on a 30 m grid whose cell edges are multiples of 30 m**. The LiDAR zoom area uses
the same lattice, so the SRTM cells line up exactly with the aggregated LiDAR cells. Earth Engine resamples SRTM from its
geographic 1-arc-second grid with **bilinear** interpolation (smoother than nearest-neighbour for a continuous surface).

#### SRTM access from Earth Engine
Here we request the `elevation` band from `USGS/SRTMGL1_003`, projected onto a **30 m UTM grid**. The original product is a near-global 1 arc-second DEM acquired in 2000, not a native 1 m terrain model. The resampled SRTM grid is used for comparison only; interpolation cannot restore small ditches and potholes absent from the original signal.

If Earth Engine requests fail, verify the project and the regional ROI first. Avoid repeated concurrent requests or arbitrarily increasing the ROI.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | DOWNLOAD AND ALIGN SRTM FROM GEE
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

def srtm_grid_bounds(b, pad=2):
    xmin = math.floor(b[0] / GRID) * GRID - pad * GRID; ymin = math.floor(b[1] / GRID) * GRID - pad * GRID
    xmax = math.ceil(b[2] / GRID) * GRID + pad * GRID;  ymax = math.ceil(b[3] / GRID) * GRID + pad * GRID
    return xmin, ymin, xmax, ymax

def download_srtm(bounds, out_path):
    if out_path.exists():
        print('Reusing cached SRTM:', out_path.name); return out_path
    xmin, ymin, xmax, ymax = bounds
    w, h = int(round((xmax - xmin) / GRID)), int(round((ymax - ymin) / GRID))
    img = ee.Image('USGS/SRTMGL1_003').select('elevation').resample('bilinear').toFloat()
    url = img.getDownloadURL({'crs': CRS, 'crs_transform': [GRID, 0, xmin, 0, -GRID, ymax],
                              'dimensions': f'{w}x{h}', 'format': 'GEO_TIFF'})
    print(f'Downloading SRTM from Earth Engine ({w} x {h} cells at 30 m) ...')
    r = requests.get(url, timeout=600); r.raise_for_status()
    tmp = out_path.with_suffix('.download')
    tmp.write_bytes(r.content)
    if zipfile.is_zipfile(tmp):
        with zipfile.ZipFile(tmp) as z:
            name = [n for n in z.namelist() if n.endswith('.tif')][0]
            out_path.write_bytes(z.read(name))
        tmp.unlink()
    else:
        tmp.rename(out_path)
    return out_path

SRTM_BOUNDS = srtm_grid_bounds(ROI_UTM.total_bounds)
srtm_raw_path = download_srtm(SRTM_BOUNDS, CACHE / 'SRTM_ROI_raw.tif')
srtm_raw, srtm_tr = read_raster(srtm_raw_path)
with rasterio.open(srtm_raw_path) as s:
    assert s.crs.to_epsg() == CRS_EPSG and abs(s.res[0] - GRID) < 1e-6, 'Unexpected SRTM grid'
print(f'SRTM grid: {srtm_raw.shape[1]} x {srtm_raw.shape[0]} cells, origin ({srtm_tr.c:.0f}, {srtm_tr.f:.0f}), '
      f'elevation {np.nanmin(srtm_raw):.1f}-{np.nanmax(srtm_raw):.1f} m')

## 7. Preparing the common 30 m comparison grid (DEM C) and the common footprint
* **DEM C (LiDAR 30 m)** is computed as the **area-weighted mean** of the 900 one-metre cells inside each 30 m cell. Because the
  1 m cells nest exactly inside the 30 m cells, this is a simple block mean. A 30 m cell is kept only if at least 90 % of its
  1 m cells are valid.
* **DEM B (SRTM 30 m)** for the zoom area is cut from the whole-ROI SRTM grid (same lattice, no resampling).
* All three DEMs are then restricted to the **same valid footprint**: the analysis polygon, at the resolution of the 30 m
  cells that lie (≥ 90 %) inside it. A cell is analysed only where all three products have data.
* The **whole-ROI SRTM** is masked to the ROI polygon for the regional analyses.

**LiDAR 1 m stays at 1 m** in every analysis. It is never presented as if it were a 30 m product.

**ROI geometry validation (Version 4):** The regional mask is created from all features in `ROI_1.shp`, projected to EPSG:26914, and applied to the whole-ROI SRTM surface. The code now defines `ROI_GEOM` both in the ROI-loading step and locally in the DEM preparation step so that rerunning the latter does not cause a missing-variable error.

#### DEM-to-DEM comparison: distinguish resolution from sensor differences
Three working DEMs cover the *same geographic footprint*. **LiDAR 1 m → LiDAR 30 m** isolates the consequences of aggregation. **LiDAR 30 m ↔ SRTM 30 m** compares different elevation sources at the same displayed grid spacing. Summary differences should use matching valid pixels and explicit vertical-reference caveats; SRTM and LiDAR elevations are not necessarily directly comparable without vertical-datum checks.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | PREPARE THE 1 m AND MATCHED 30 m COMPARISON DEMs
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

zb = ZOOM_BOUNDS
lidar_1m, tr_1m = read_raster(LIDAR_1M_PATH)
lidar_1m[geometry_mask([ANALYSIS_POLY], lidar_1m.shape, tr_1m)] = np.nan     # keep only the analysis polygon
# DEM C: LiDAR aggregated to 30 m
lidar_30 = block_mean(lidar_1m, AGG, min_valid=0.9)
tr_30 = from_origin(zb[0], zb[3], GRID, GRID)
# DEM B: SRTM for the zoom area, cut from the same 30 m lattice
c0 = int(round((zb[0] - srtm_tr.c) / GRID)); r0 = int(round((srtm_tr.f - zb[3]) / GRID))
srtm_30 = srtm_raw[r0:r0 + lidar_30.shape[0], c0:c0 + lidar_30.shape[1]].copy()
assert srtm_30.shape == lidar_30.shape, 'SRTM and LiDAR 30 m grids do not line up'
# Common footprint (valid in all three)
common30 = np.isfinite(lidar_30) & np.isfinite(srtm_30)
lidar_30[~common30] = np.nan; srtm_30[~common30] = np.nan
lidar_1m[~upsample(common30)] = np.nan
# Whole-ROI SRTM: mask everything OUTSIDE the supplied regional polygon.
# Build ROI_GEOM directly from the GeoDataFrame here as well, so this processing
# cell works even in a Colab session that loaded an older ROI-setup cell.
ROI_GEOM = (ROI_UTM.geometry.union_all()
            if hasattr(ROI_UTM.geometry, 'union_all')
            else ROI_UTM.geometry.unary_union)
if not ROI_GEOM.is_valid:
    ROI_GEOM = ROI_GEOM.buffer(0)
if ROI_GEOM.is_empty:
    raise ValueError('Regional ROI geometry is empty: rerun the boundary-loading cell.')
srtm_roi = srtm_raw.copy()
outside_roi = geometry_mask([ROI_GEOM], out_shape=srtm_roi.shape,
                            transform=srtm_tr, invert=False)
srtm_roi[outside_roi] = np.nan
if not np.isfinite(srtm_roi).any():
    raise RuntimeError('The regional ROI does not overlap the downloaded SRTM grid. Check CRS and boundaries.')

TRANSFORM = {'LiDAR_1m': tr_1m, 'LiDAR_30m': tr_30, 'SRTM_30m': tr_30, 'SRTM_ROI': srtm_tr}
ARR = {'LiDAR_1m': lidar_1m, 'LiDAR_30m': lidar_30, 'SRTM_30m': srtm_30, 'SRTM_ROI': srtm_roi}
SHAPE = {k: v.shape for k, v in ARR.items()}
DEM_PATH = {}
for t, a in ARR.items():          # plain (uncompressed) GeoTIFFs for WhiteboxTools
    DEM_PATH[t] = write_raster(WORK / MODE / f'dem_{t}.tif', a, TRANSFORM[t], compress=False)
HS = {t: np.where(np.isfinite(ARR[t]), hillshade(ARR[t], RES[t]), np.nan) for t in ARR}

grid_df = pd.DataFrame([{'product': LABEL[t], 'cell_m': RES[t], 'columns': SHAPE[t][1], 'rows': SHAPE[t][0],
                         'valid_cells': int(np.isfinite(ARR[t]).sum()),
                         'valid_area_ha': np.isfinite(ARR[t]).sum() * RES[t] ** 2 / 1e4,
                         'x_origin': TRANSFORM[t].c, 'y_origin': TRANSFORM[t].f,
                         'elev_min_m': np.nanmin(ARR[t]), 'elev_max_m': np.nanmax(ARR[t])} for t in ARR])
show_table(grid_df.set_index('product'), 'grids_and_footprint.csv', OUT['S'], 2)
ZOOM_AREA_HA = grid_df.loc[0, 'valid_area_ha']

### 8.0 Run the hydrological workflow for every DEM
The same WhiteboxTools workflow runs on all four DEMs (three zoom-area products + whole-ROI SRTM). **Two surfaces are
kept separate on purpose:**

| Surface | Used for | Why |
|---|---|---|
| **Original DEM** (unfilled) | Slope, aspect, depressions and potential storage | Prairie potholes are real storage features. Filling them would erase the water we want to measure. |
| **Conditioned DEM** (`CONDITIONING = "fill"`: depressions filled, flats given a tiny gradient) | Flow direction, flow accumulation, streams, watersheds, TWI | D8 routing needs every cell to drain somewhere. Filling represents **fill-and-spill**: water routes *through* a depression only after it is full. |
| **Depression-filled surface without gradients** | Depth of each depression and its **spill elevation** | depth = filled − original |

💡 `CONDITIONING = "breach"` instead cuts narrow channels through barriers (for example road embankments that may hide
culverts). You can investigate this setting in an optional rerun after finishing the eight analyses.

#### Why terrain conditioning is necessary
A D8 flow algorithm assumes each cell can route downslope water. Yet real prairie potholes are closed and may hold water for long periods. For routing, WhiteboxTools fills or breaches obstructing sinks, according to `CONDITIONING`. For storage, **the original DEM is kept unchanged**. Filling the DEM for drainage calculations does **not** mean we assume real wetlands always spill.

**Warning:** D8 is a *single-flow-direction* model. In broad flat areas, multiple flow directions, culverts, subsurface drainage and infiltration may matter, but are not represented here.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | CONDITION ROUTING DEMs WHILE PRESERVING RAW SURFACES
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

print(f'Hydrological workflow (conditioning = {CONDITIONING}) ...')
HY = {t: run_hydrology(t, DEM_PATH[t]) for t in ['LiDAR_1m', 'LiDAR_30m', 'SRTM_30m', 'SRTM_ROI']}
_LOADED.clear()

## 8. The eight hydrological analyses
---
## Analysis 1 — Elevation and slope mapping
**Principle.** Slope (the steepest local gradient) and aspect (the direction the slope faces) control how fast and in which
direction water moves over the surface. They are computed from the elevation differences between each cell and its eight
neighbours, so they depend directly on the DEM's pixel size and noise.

**Question.** How well does each DEM represent the gentle slopes and subtle landforms (beach ridges, the creek valley, ditches,
depressions) of a prairie landscape, and how large are the elevation differences between products on a common grid?

**Processing notes (Analysis 1).** Elevation values are in metres; slope maps use degrees. Compare hillshades under equal illumination rather than auto-stretching each image. At very low slopes, aspect can be unstable and may not have an interpretable downslope direction. Summarize SRTM-minus-LiDAR bias and RMSE only on aligned, valid cells.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 1: ELEVATION / SLOPE / ASPECT
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# Compare each DEM on the same footprint before interpreting relief and slope.
# Bias/MAE/RMSE require aligned pixels and consistent vertical height references.
E = {t: R(t, 'dem') for t in PRODUCTS}
SL = {t: R(t, 'slope') for t in PRODUCTS}
AS = {t: R(t, 'aspect') for t in PRODUCTS}
vlo, vhi = np.nanpercentile(np.concatenate([sample_values(E[t]) for t in PRODUCTS]), [1, 99])
compare_maps(E, 'Analysis 1a — Elevation (identical colour scale)', 'Elevation (m)', 'terrain', vlo, vhi,
             fname='A1a_elevation_maps.png', folder=OUT['01'])
compare_maps(SL, 'Analysis 1b — Slope (display capped at 3°; LiDAR ditches reach much higher)', 'Slope (degrees)',
             'magma_r', 0, 3, fname='A1b_slope_maps.png', folder=OUT['01'], extend='max')
compare_maps(AS, 'Analysis 1c — Aspect (direction the slope faces)', 'Aspect (degrees from north)', 'twilight', 0, 360,
             fname='A1c_aspect_maps.png', folder=OUT['01'], hill=False)

# --- statistics on the common 30 m grid ---
d_srtm = E['SRTM_30m'] - E['LiDAR_30m']                       # source difference at equal resolution
resid_1m = E['LiDAR_1m'] - upsample(E['LiDAR_30m'])           # sub-grid relief lost by aggregation
s_srtm = diff_stats(E['SRTM_30m'], E['LiDAR_30m'])
s_agg = diff_stats(E['LiDAR_1m'], upsample(E['LiDAR_30m']))
slope_1m_as30 = block_mean(SL['LiDAR_1m'])
elev_tab = pd.DataFrame({
    'SRTM 30 m − LiDAR 30 m (source effect)': s_srtm,
    'LiDAR 1 m − LiDAR 30 m (resolution effect)': s_agg}).T
show_table(elev_tab, 'A1_elevation_difference_stats.csv', OUT['01'])

def slope_stats(a):
    v = a[np.isfinite(a)]
    return {'mean_deg': v.mean(), 'median_deg': np.median(v), 'p95_deg': np.percentile(v, 95),
            'pct_area_below_0.5deg': (v < 0.5).mean() * 100, 'pct_area_below_2deg': (v < 2).mean() * 100}
slope_tab = pd.DataFrame({LABEL[t]: slope_stats(SL[t]) for t in PRODUCTS}).T
slope_tab.loc['LiDAR 1 m slope averaged to 30 m'] = slope_stats(slope_1m_as30)
show_table(slope_tab, 'A1_slope_stats.csv', OUT['01'])

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)
bins = np.linspace(0, 5, 101)
for t in PRODUCTS:
    ax[0].hist(sample_values(SL[t]), bins=bins, density=True, histtype='step', lw=2, color=COLOR[t], label=SHORT[t])
ax[0].set(xlabel='Slope (degrees)', ylabel='Density', title='Slope distributions'); ax[0].legend()
m = np.isfinite(d_srtm)
ax[1].scatter(E['LiDAR_30m'][m], E['SRTM_30m'][m], s=3, alpha=0.4, color=COLOR['SRTM_30m'])
lim = [np.nanmin(E['LiDAR_30m']), np.nanmax(E['LiDAR_30m'])]
ax[1].plot(lim, lim, 'k--', lw=1)
ax[1].set(xlabel='LiDAR 30 m elevation (m)', ylabel='SRTM 30 m elevation (m)',
          title=f"SRTM vs LiDAR 30 m: bias {s_srtm['bias_m']:+.2f} m, RMSE {s_srtm['RMSE_m']:.2f} m")
ax[2].hist(d_srtm[m], bins=60, color=COLOR['SRTM_30m'], alpha=0.7)
ax[2].axvline(0, color='k', lw=1)
ax[2].set(xlabel='SRTM − LiDAR 30 m (m)', ylabel='Cells (30 m)', title='Elevation differences')
fig.savefig(OUT['01'] / 'A1d_slope_histograms_and_elevation_scatter.png', dpi=300); plt.show()

dl = np.nanpercentile(np.abs(d_srtm), 98)
fig, ax = plt.subplots(1, 2, figsize=(12.5, 5.6), constrained_layout=True)
for a_, arr, t, ttl, lim_ in [(ax[0], d_srtm, 'SRTM_30m', 'SRTM 30 m − LiDAR 30 m', dl),
                              (ax[1], resid_1m, 'LiDAR_1m', 'LiDAR 1 m − LiDAR 30 m (sub-grid relief)', 1.0)]:
    a_.imshow(_disp(HS[t])[0], cmap='gray', extent=extent_of(t))
    im = a_.imshow(_disp(arr)[0], cmap='RdBu_r', vmin=-lim_, vmax=lim_, extent=extent_of(t), alpha=0.85)
    fig.colorbar(im, ax=a_, shrink=0.8).set_label('Elevation difference (m)')
    a_.set_title(ttl); a_.ticklabel_format(style='plain', useOffset=False); a_.tick_params(labelsize=7)
fig.suptitle('Analysis 1e — Elevation-difference maps (note the different colour ranges)')
fig.savefig(OUT['01'] / 'A1e_elevation_difference_maps.png', dpi=300); plt.show()

regional_map(R('SRTM_ROI', 'dem'), 'Whole ROI — SRTM elevation', 'Elevation (m)', 'terrain',
             fname='A1_regional_SRTM_elevation.png', folder=OUT['01'])
regional_map(R('SRTM_ROI', 'slope'), 'Whole ROI — SRTM slope', 'Slope (degrees)', 'magma_r', 0, 3,
             fname='A1_regional_SRTM_slope.png', folder=OUT['01'])

for t in PRODUCTS:
    export_raster(E[t], t, 'A1_elevation', OUT['01']); export_raster(SL[t], t, 'A1_slope_deg', OUT['01'])
    export_raster(AS[t], t, 'A1_aspect_deg', OUT['01'])
export_raster(d_srtm, 'SRTM_30m', 'A1_diff_SRTM_minus_LiDAR30', OUT['01'])
add_summary('1 Elevation & slope', 'Mean slope', 'degrees', {t: np.nanmean(SL[t]) for t in PRODUCTS})
add_summary('1 Elevation & slope', 'Area with slope < 0.5°', '%', {t: (SL[t][np.isfinite(SL[t])] < 0.5).mean() * 100 for t in PRODUCTS})
add_summary('1 Elevation & slope', 'Elevation RMSE vs LiDAR 30 m', 'm',
            {'LiDAR_1m': s_agg['RMSE_m'], 'SRTM_30m': s_srtm['RMSE_m']})
add_summary('1 Elevation & slope', 'Elevation bias vs LiDAR 30 m', 'm',
            {'LiDAR_1m': s_agg['bias_m'], 'SRTM_30m': s_srtm['bias_m']})

**Interpretation guide — evaluate the generated maps and tables (do not assume the outcome).**
* **Elevation:** Compare the extent and magnitude of SRTM − LiDAR 30 m differences and report the observed bias, MAE and RMSE. Potential explanations include vertical-datum differences, radar surface scattering, noise, resampling, and real terrain changes between acquisition dates. Do not attribute the differences to a particular cause without supporting evidence.
* **Slope:** Examine which ditches, ridges and road embankments are visible in native LiDAR but less distinct at 30 m. Coarsening often smooths steep microtopographic gradients, while source noise may introduce spurious local slopes.
* **Controlled comparison:** A vs C examines grid coarsening within LiDAR; B vs C compares co-registered LiDAR 30 m and SRTM 30 m, which still differ in sensing method, acquisition and processing.

✏️ **Questions**
1. Which landforms visible in the 1 m hillshade disappear in LiDAR 30 m? Which remain recognizable?
2. Where are the largest elevation differences, and what independent evidence could distinguish bias, land-cover effects, and actual terrain change?


---
## Analysis 2 — Flow direction (D8)
**Principle.** The **D8** method sends all water from a cell to the **one** of its eight neighbours with the steepest downhill
drop. D8 is simple and consistent, but it needs every cell to have a lower neighbour. **Pits, closed depressions and flat
areas** have none, so D8 is computed on the **conditioned DEM**. There, depressions are filled to their spill level and
flats receive a tiny gradient towards the outlet. The direction of water *inside* a filled depression is therefore a product of
conditioning: in reality the water is stored there until the depression overflows (natural **closed drainage**).

**Question.** Do the three DEMs send water in the same directions, and how much of each landscape has a flow direction that
is only an artefact of hydrological conditioning?

**Reading the maps:** Each cell's colour represents one of the eight D8 directions. **Black arrows show the direction of flow** (NE, E, SE, S, SW, W, NW or N), as in the class example. For legibility, arrows are **sampled** at regular map intervals; the computations still use *every* valid cell. Arrows represent local flow directions, not measured runoff or continuous stream paths.


**Processing notes (Analysis 2).** The Whitebox D8 pointer records one of eight downstream directions per cell. The compass legend must use the pointer coding implemented by the chosen tool—not an arbitrary numeric ordering. Compare directional agreement on comparable grids, and recognize that near-flat cells can flip directions due to small height differences.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 2: D8 FLOW DIRECTION
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# D8 routes each pixel to ONE steepest downslope neighbour (a simplification).
# Verify the Whitebox pointer scheme when relating code values to compass directions.
P = {t: R(t, 'pntr') for t in PRODUCTS}
d8_codes = list(D8)
d8_cmap = ListedColormap(['#e41a1c', '#ff7f00', '#ffff33', '#4daf4a', '#a6d854', '#377eb8', '#984ea3', '#f781bf'])
d8_norm = BoundaryNorm([0.5] + [k * 1.5 for k in d8_codes], 8)
P_idx = {t: np.where(np.isin(P[t], d8_codes), np.log2(np.nan_to_num(P[t], nan=1)).astype(float), np.nan) for t in PRODUCTS}
idx_norm = BoundaryNorm(np.arange(-0.5, 8.5, 1), 8)
legend = [Patch(color=d8_cmap(i), label=D8[k][2]) for i, k in enumerate(d8_codes)]
legend.append(Line2D([], [], color='#111111', marker=r'$\rightarrow$', linestyle='None',
                     markersize=12, label='D8 flow arrow (sampled)'))

def show_flow_arrows(ax, t):
    # Black arrows are the actual D8 directions at regularly spaced sample cells.
    # Use common map-unit spacing on both resolutions for visual comparability.
    n = overlay_d8_arrows(ax, P[t], TRANSFORM[t], spacing_m=240, color='black')
    ax.text(0.02, 0.02, f'{n} sampled arrows', transform=ax.transAxes, fontsize=7,
            va='bottom', ha='left', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
compare_maps(P_idx, 'Analysis 2a — D8 flow direction on the conditioned DEM', '', d8_cmap, norm=idx_norm,
             fname='A2a_flow_direction_maps.png', folder=OUT['02'], legend=legend, alpha=0.65,
             overlay=show_flow_arrows)

# Where did conditioning change the surface? (cond - original > 0)
MOD = {t: np.where(np.isfinite(E[t]), (R(t, 'cond') - E[t]) > 1e-5, np.nan).astype('float32') for t in PRODUCTS}
compare_maps(MOD, 'Analysis 2b — Cells raised by conditioning (flow direction here is a conditioning artefact)',
             '1 = raised', ListedColormap(['#ffffff00', '#08519c']), 0, 1, fname='A2b_conditioning_modified.png',
             folder=OUT['02'], legend=[Patch(color='#08519c', label='raised by depression filling / flat removal')])

rows = {}
for t in PRODUCTS:
    v = P[t][np.isfinite(P[t])]
    freq = {f'% {D8[k][2]}': (v == k).mean() * 100 for k in d8_codes}
    rows[LABEL[t]] = {'% area modified by conditioning': np.nanmean(MOD[t]) * 100,
                      '% cells without receiver (outlets)': (v == 0).mean() * 100, **freq}
dir_tab = pd.DataFrame(rows).T
show_table(dir_tab, 'A2_flow_direction_stats.csv', OUT['02'], 2)

# Agreement on the 30 m grid
same_src = np.isfinite(P['LiDAR_30m']) & np.isfinite(P['SRTM_30m'])
agree_src = (P['LiDAR_30m'][same_src] == P['SRTM_30m'][same_src]).mean() * 100
# dominant 1 m direction inside each 30 m cell
h30, w30 = SHAPE['LiDAR_30m']
counts = np.stack([block_mean((P['LiDAR_1m'] == k).astype('float32'), AGG, 0) for k in d8_codes])
dominant = np.array(d8_codes)[counts.argmax(0)]
agree_res = (dominant[same_src] == P['LiDAR_30m'][same_src]).mean() * 100
# Independence baseline based on the *observed* frequency of each direction (not a fixed 12.5%).
# A uniform 1/8 baseline is usually misleading because flow directions are not equally frequent.
def independent_agreement(x, y, mask):
    a, b = x[mask], y[mask]
    return 100 * sum((a == k).mean() * (b == k).mean() for k in d8_codes)
chance_src = independent_agreement(P['LiDAR_30m'], P['SRTM_30m'], same_src)
chance_res = independent_agreement(dominant, P['LiDAR_30m'], same_src)
agree_tab = pd.DataFrame({'agreement_%': [agree_src, chance_src, agree_res, chance_res]},
                         index=['LiDAR 30 m vs SRTM 30 m (same cell)',
                                'Independence baseline for those products (observed marginals)',
                                'Dominant LiDAR 1 m direction vs LiDAR 30 m',
                                'Independence baseline for resolution comparison'])
show_table(agree_tab, 'A2_flow_direction_agreement.csv', OUT['02'], 1)

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(5.5, 5.5))
ang = np.radians([45, 90, 135, 180, 225, 270, 315, 0])
for t in PRODUCTS:
    f_ = [dir_tab.loc[LABEL[t], f'% {D8[k][2]}'] for k in d8_codes]
    ax.plot(np.append(ang, ang[0]), np.append(f_, f_[0]), color=COLOR[t], lw=2, label=SHORT[t])
ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
ax.set_title('Share of cells by flow direction (%)'); ax.legend(loc='lower right', fontsize=8, bbox_to_anchor=(1.25, -0.05))
fig.savefig(OUT['02'] / 'A2c_flow_direction_rose.png', dpi=300, bbox_inches='tight'); plt.show()

pr = R('SRTM_ROI', 'pntr')
regional_map(np.where(np.isin(pr, d8_codes), np.log2(np.nan_to_num(pr, nan=1)), np.nan),
             'Whole ROI — SRTM D8 flow direction', '', d8_cmap, norm=idx_norm, fname='A2_regional_SRTM_flowdir.png',
             folder=OUT['02'], legend=legend,
             overlay=lambda ax: overlay_d8_arrows(ax, pr, TRANSFORM['SRTM_ROI'],
                                                   spacing_m=1800, color='black'))
for t in PRODUCTS:
    export_raster(P[t], t, 'A2_d8_pointer_whitebox', OUT['02'])
add_summary('2 Flow direction', 'Area modified by conditioning', '%', {t: np.nanmean(MOD[t]) * 100 for t in PRODUCTS})
add_summary('2 Flow direction', 'D8 agreement with LiDAR 30 m', '%', {'LiDAR_1m': agree_res, 'SRTM_30m': agree_src})

**Interpretation guide — test the modelled D8 directions.**
* In nearly flat terrain, small elevation differences can change the selected downslope neighbour. Quantify directional agreement rather than presuming it is high or low. The comparison table also reports an **independence baseline based on the observed direction frequencies**.
* The black arrows point toward the D8 receiver of sampled cells. Their purpose is visual interpretation; the results are calculated from all cells. These arrows are not measurements of flow or stream-channel vectors.
* Where depression filling changes the DEM, flow directions are produced partly by **conditioning**. Such routing represents potential overflow pathways after filling, not uninterrupted natural runoff through a pothole.
* Road embankments and missing culverts can modify the apparent drainage direction. Compare the conditioning map with roads, ditches and terrain hillshade.

✏️ **Questions**
1. Which landscape features correspond to the largest areas of DEM conditioning? Are they natural depressions, ditches, or possible artifacts?
2. Does the observed D8 agreement exceed its directional-frequency baseline? Why does agreement vary across the three DEM products?


---
## Analysis 3 — Flow accumulation and drainage pathways
**Principle.** Flow accumulation counts how many upslope cells drain through each cell. Multiplied by the cell area it gives
the **upstream contributing area** (here in hectares), which is independent of pixel size and so comparable between DEMs.
Cells whose contributing area exceeds a threshold (`STREAM_THRESHOLD_HA`) are mapped as **potential drainage pathways**
(channels, ditches, swales). These are pathways water *would* follow if the depressions were full; they are not observed
streams.

**Question.** How do resolution and data source change the density and position of drainage pathways?

**Processing notes (Analysis 3).** Accumulated cell counts are converted to physical **contributing area (hectares)** before comparison. Use a single threshold in hectares for all datasets. The mapped drainage pathways are *topographic predictions*, not evidence that a flowing channel existed during image acquisition.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 3: CONTRIBUTING AREA / DRAINAGE
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# Convert upstream cell count into a comparable PHYSICAL AREA:
# 1 m cell = 1 m²; 30 m cell = 900 m²; 10,000 m² = 1 ha.
ACC = {t: R(t, 'acc_cells') * RES[t] ** 2 / 1e4 for t in PRODUCTS}          # contributing area (ha)
LACC = {t: np.log10(np.maximum(ACC[t], 1e-4)) for t in PRODUCTS}
STREAMS = {}
for t in PRODUCTS + ['SRTM_ROI']:
    g = gpd.read_file(HY[t]['streams_shp'])
    STREAMS[t] = g.set_crs(CRS, allow_override=True)
def stream_overlay(ax, t):
    if len(STREAMS[t]):
        STREAMS[t].plot(ax=ax, color='#08306b', lw=0.8 if t == 'LiDAR_1m' else 1.3)
compare_maps(LACC, 'Analysis 3a — Upstream contributing area (log scale)', 'log10(contributing area, ha)', 'Blues',
             -2, 3, fname='A3a_flow_accumulation_maps.png', folder=OUT['03'], hill=True)
compare_maps({t: np.where(np.isfinite(E[t]), np.nan, np.nan) for t in PRODUCTS},
             f'Analysis 3b — Potential drainage pathways (contributing area ≥ {STREAM_THRESHOLD_HA} ha)', '',
             fname='A3b_drainage_pathways.png', folder=OUT['03'], overlay=stream_overlay,
             legend=[Line2D([], [], color='#08306b', label=f'pathway (≥ {STREAM_THRESHOLD_HA} ha)')])

area_km2 = {t: np.isfinite(E[t]).sum() * RES[t] ** 2 / 1e6 for t in PRODUCTS}
length_km = {t: STREAMS[t].length.sum() / 1000 for t in PRODUCTS}
dd = {t: length_km[t] / area_km2[t] for t in PRODUCTS}
# positional agreement: share of LiDAR 1 m pathway cells within 45 m (1.5 cells) of a 30 m pathway
s1 = R('LiDAR_1m', 'streams') > 0
within = {}
for t in ['LiDAR_30m', 'SRTM_30m']:
    dist = ndimage.distance_transform_edt(~upsample(R(t, 'streams') > 0))
    within[t] = (dist[s1] <= 45).mean() * 100 if s1.any() else np.nan
acc_tab = pd.DataFrame({LABEL[t]: {'pathway_length_km': length_km[t], 'drainage_density_km_per_km2': dd[t],
                                   'max_contributing_area_ha': np.nanmax(ACC[t]),
                                   '% of LiDAR 1 m pathways within 45 m': within.get(t, 100.0)} for t in PRODUCTS}).T
show_table(acc_tab, 'A3_flow_accumulation_stats.csv', OUT['03'], 2)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for t in PRODUCTS:
    v = np.sort(ACC[t][np.isfinite(ACC[t])])
    ex = 1 - np.arange(v.size) / v.size
    k = max(1, v.size // 5000)
    ax[0].loglog(v[::k], ex[::k], color=COLOR[t], lw=2, label=SHORT[t])
ax[0].axvline(STREAM_THRESHOLD_HA, color='k', ls='--', lw=1)
ax[0].set(xlabel='Contributing area (ha)', ylabel='Fraction of cells with ≥ this area',
          title='Contributing-area exceedance curves'); ax[0].legend()
ax[1].bar([SHORT[t] for t in PRODUCTS], [dd[t] for t in PRODUCTS], color=[COLOR[t] for t in PRODUCTS])
ax[1].set(ylabel='Drainage density (km per km²)', title=f'Pathway density (threshold {STREAM_THRESHOLD_HA} ha)')
fig.savefig(OUT['03'] / 'A3c_contributing_area_distribution.png', dpi=300); plt.show()

regional_map(np.log10(np.maximum(R('SRTM_ROI', 'acc_cells') * 0.09, 1e-2)), 'Whole ROI — SRTM contributing area',
             'log10(ha)', 'Blues', -1.5, 4, fname='A3_regional_SRTM_accumulation.png', folder=OUT['03'],
             overlay=lambda ax: STREAMS['SRTM_ROI'].plot(ax=ax, color='#08306b', lw=0.6))
for t in PRODUCTS + ['SRTM_ROI']:
    STREAMS[t].assign(product=SHORT[t]).to_file(OUT['03'] / 'A3_drainage_pathways.gpkg', layer=t, driver='GPKG')
for t in PRODUCTS:
    export_raster(ACC[t], t, 'A3_contributing_area_ha', OUT['03'])
add_summary('3 Flow accumulation', 'Drainage density', 'km/km²', dd)
add_summary('3 Flow accumulation', 'LiDAR 1 m pathways within 45 m', '%', within)

**Interpretation guide.**
* Compare mapped pathways using a physically consistent contributing-area threshold in hectares. The number of accumulated cells alone is **not** comparable between 1 m and 30 m grids.
* At coarser resolution, narrow ditches and small swales may be smoothed or displaced; however, major pathways may remain recognizable. Check the maps to determine what actually happened in Elm Creek.
* Source differences and DEM conditioning can redirect pathways, especially on gentle prairie terrain. Test whether paths correspond to visible valleys, ditches and roads.
* Optional experiment: compare thresholds such as `STREAM_THRESHOLD_HA = 2`, `10`, and `50` ha (rerun the relevant workflow).

✏️ **Questions**
1. Which drainage paths look natural versus artificial (e.g., parallel to roads)? Which are captured by all three DEMs?
2. Why is contributing area in hectares a fairer comparison than raw contributing-cell counts?


---
## Analysis 4 — Watershed delineation
**Principle.** A watershed (catchment) is all the land that drains to one outlet. It is delineated by following the D8
directions upstream from the outlet. The outlet is first **snapped** to the cell with the **largest flow accumulation
within `SNAP_DISTANCE_M`** (the same tolerance for every DEM; WhiteboxTools `SnapPourPoints`), because a point that misses
the channel by one cell would produce a tiny catchment. **Subcatchments** are the areas draining to each pathway segment.

**Default outlet** (`OUTLET_X, OUTLET_Y = 569450, 5508000`): the small incised gully that cuts through the beach ridge in
the centre of the zoom area (clearly visible in the 1 m hillshade). Water reaches it from the higher lake-plain surface to
the west, so its catchment extends **beyond the zoom-area edge and is truncated** there. The truncation is the same for all
three DEMs, so they remain comparable, and the table flags it. ✏️ Choose your own outlet on another pathway from the
Analysis 3 map (for example a small catchment that lies completely inside the area).

**Question.** For the same outlet, how different are the catchment boundary and area derived from the three DEMs?

**Processing notes (Analysis 4).** Watershed boundaries depend on the **outlet**. The code attempts to snap the given outlet to high-accumulation cells within `SNAP_DISTANCE_M`; inspect the snapped coordinates for all three DEMs before interpreting boundary differences. A different snapped outlet can dominate the apparent DEM comparison.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 4: WATERSHED AND OUTLET
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# Catchment boundaries are outlet-sensitive; compare snapped outlet coordinates first.
# An inconsistent outlet can explain more disagreement than DEM grid size.
WS, SNAP = {}, {}
for t in PRODUCTS + ['SRTM_ROI']:
    w = Path(HY[t]['pntr']).parent
    gpd.GeoDataFrame({'id': [1]}, geometry=[Point(OUTLET_X, OUTLET_Y)], crs=CRS).to_file(w / 'outlet.shp')
    wbt_run('snap_pour_points', pour_pts=str(w / 'outlet.shp'), flow_accum=HY[t]['acc_cells'],
            output=str(w / 'outlet_snapped.shp'), snap_dist=SNAP_DISTANCE_M)
    sp = gpd.read_file(w / 'outlet_snapped.shp').set_crs(CRS, allow_override=True)
    wbt_run('watershed', d8_pntr=HY[t]['pntr'], pour_pts=str(w / 'outlet_snapped.shp'), output=str(w / 'watershed.tif'))
    ws, _ = read_raster(w / 'watershed.tif')
    WS[t] = np.isfinite(ws) & (ws > 0)
    SNAP[t] = sp.geometry.iloc[0]
SUB = {t: R(t, 'subbasins') for t in PRODUCTS}

def ws_polygon(mask, t):
    geoms = [shape(g) for g, v in shapes(mask.astype('uint8'), mask=mask, transform=TRANSFORM[t])]
    from shapely.ops import unary_union
    return unary_union(geoms) if geoms else None
WS_POLY = {t: ws_polygon(WS[t], t) for t in PRODUCTS + ['SRTM_ROI']}

def ws_overlay(ax, t):
    for u in PRODUCTS:
        if WS_POLY[u] is not None:
            gpd.GeoSeries([WS_POLY[u]], crs=CRS).boundary.plot(ax=ax, color=COLOR[u], lw=2.2 if u == t else 0.9,
                                                               ls='-' if u == t else '--')
    if len(STREAMS[t]):
        STREAMS[t].plot(ax=ax, color='#08306b', lw=0.6)
    ax.plot(OUTLET_X, OUTLET_Y, 'kx', ms=9, mew=2)
    ax.plot(SNAP[t].x, SNAP[t].y, 'o', mfc='yellow', mec='k', ms=8)
sub_disp = {t: np.where(SUB[t] > 0, (SUB[t] * 7919) % 20, np.nan) for t in PRODUCTS}
compare_maps(sub_disp, 'Analysis 4a — Subcatchments (colours) and the watershed of the common outlet', '',
             'tab20', 0, 19, fname='A4a_watersheds.png', folder=OUT['04'], overlay=ws_overlay, alpha=0.45,
             legend=[Line2D([], [], color=COLOR[u], lw=2, label=f'watershed {SHORT[u]}') for u in PRODUCTS] +
                    [Line2D([], [], marker='x', color='k', ls='', label='outlet given'),
                     Line2D([], [], marker='o', mfc='yellow', mec='k', ls='', label='outlet after snapping')])

# areas, snapping and overlap (IoU computed on the 1 m grid; 30 m masks are expanded exactly)
M1 = {'LiDAR_1m': WS['LiDAR_1m'], 'LiDAR_30m': upsample(WS['LiDAR_30m']), 'SRTM_30m': upsample(WS['SRTM_30m'])}
def iou(a, b):
    u = (a | b).sum()
    return (a & b).sum() / u if u else np.nan
frame1 = np.zeros(SHAPE['LiDAR_1m'], bool); frame1[[0, -1], :] = True; frame1[:, [0, -1]] = True
frame1 |= ndimage.binary_dilation(~np.isfinite(E['LiDAR_1m']), structure=np.ones((3, 3)))
ws_rows = {}
for t in PRODUCTS:
    ws_rows[LABEL[t]] = {'snap_distance_m': Point(OUTLET_X, OUTLET_Y).distance(SNAP[t]),
                         'snapped_x': SNAP[t].x, 'snapped_y': SNAP[t].y,
                         'watershed_area_ha': WS[t].sum() * RES[t] ** 2 / 1e4,
                         'n_subcatchments': len(np.unique(SUB[t][np.isfinite(SUB[t]) & (SUB[t] > 0)])),
                         'touches_zoom_edge (truncated)': bool((M1[t] & frame1).any()),
                         'IoU_with_LiDAR_1m': iou(M1[t], M1['LiDAR_1m']),
                         'IoU_with_LiDAR_30m': iou(M1[t], M1['LiDAR_30m']),
                         'IoU_with_SRTM_30m': iou(M1[t], M1['SRTM_30m'])}
ws_tab = pd.DataFrame(ws_rows).T
show_table(ws_tab, 'A4_watershed_comparison.csv', OUT['04'], 3)
if ws_tab['snap_distance_m'].astype(float).max() - ws_tab['snap_distance_m'].astype(float).min() > 30:
    print('⚠️  The outlet snapped to different places on different DEMs: the DEMs place the channel differently.')

regional_map(np.where(R('SRTM_ROI', 'subbasins') > 0, (R('SRTM_ROI', 'subbasins') * 7919) % 20, np.nan),
             'Whole ROI — SRTM subcatchments', '', 'tab20', 0, 19, fname='A4_regional_SRTM_subcatchments.png',
             folder=OUT['04'], overlay=lambda ax: STREAMS['SRTM_ROI'].plot(ax=ax, color='#08306b', lw=0.5),
             legend=[Patch(color='#cccccc', label='colours = subcatchments')])
gpd.GeoDataFrame({'product': [SHORT[t] for t in PRODUCTS + ['SRTM_ROI']],
                  'area_ha': [WS[t].sum() * RES[t] ** 2 / 1e4 for t in PRODUCTS + ['SRTM_ROI']]},
                 geometry=[WS_POLY[t] for t in PRODUCTS + ['SRTM_ROI']], crs=CRS).to_file(
    OUT['04'] / 'A4_watersheds.gpkg', layer='watersheds', driver='GPKG')
gpd.GeoDataFrame({'product': [SHORT[t] for t in PRODUCTS]}, geometry=[SNAP[t] for t in PRODUCTS], crs=CRS).to_file(
    OUT['04'] / 'A4_watersheds.gpkg', layer='snapped_outlets', driver='GPKG')
for t in PRODUCTS:
    export_raster(SUB[t], t, 'A4_subcatchments', OUT['04'])
add_summary('4 Watershed', 'Watershed area (common outlet)', 'ha', {t: ws_rows[LABEL[t]]['watershed_area_ha'] for t in PRODUCTS})
add_summary('4 Watershed', 'Watershed IoU with LiDAR 1 m', '-', {t: ws_rows[LABEL[t]]['IoU_with_LiDAR_1m'] for t in PRODUCTS})

**Interpretation guide.**
* Compare catchment area and spatial overlap (Intersection over Union, IoU) at a common outlet. Interpret IoU along with the actual boundaries: a single number cannot explain where the disagreement occurs.
* Flat divides can be sensitive to small elevation perturbations; roads, ditches, culverts and outlet snapping can also alter delineated drainage areas.
* **Do not assume** that LiDAR 30 m or SRTM better matches LiDAR 1 m at the outlet. Compare the printed coordinates of the snapped outlets and the corresponding watershed areas first.
* Subcatchments defined by grid routing do not by themselves establish which lands contribute runoff in every event; closed depressions may disconnect areas until they fill.

✏️ **Questions**
1. Where along the three watershed boundaries do differences concentrate? Are the areas flat, ditched, or near an embankment?
2. How would moving the selected outlet, or changing the snapping tolerance, affect the comparison?


---
## Analysis 5 — Topographic Wetness Index (TWI)
**Principle.** Water tends to accumulate where a large area drains in (**a**) and the ground is too flat to move it on
quickly (**β**, the local slope). The TWI combines both:

> **TWI = ln( a / tan β )**
> **a** = specific contributing area (upslope area per metre of contour width, m), **β** = local slope angle, **ln** = natural log.

High TWI means a high **potential** for wetness. The local slope comes from the original DEM. The contributing area comes from
the conditioned routing. **Numerical safeguard:** on perfectly flat cells tan β = 0 and the index would be infinite, so tan β
is limited to at least 0.001 (≈ 0.06°). Cells at that limit are reported. **TWI is a topographic index of wetness potential,
not a measurement of soil moisture.**

**Question.** How does DEM resolution change where the landscape appears "wet", and how sensitive is TWI to the flat-slope
safeguard in prairie terrain?

**Processing notes (Analysis 5).** TWI is defined from specific contributing area and local slope and is **dimensionless**. It becomes unstable as slope approaches zero, so the notebook imposes `TAN_MIN=0.001`. Report the percentage of cells affected by that floor: high TWI is *potential* wetness, not measured root-zone moisture.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 5: TOPOGRAPHIC WETNESS INDEX
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# TWI = ln(specific contributing area / tan(slope)); small slopes need a numerical floor.
# Always report the percentage of cells affected by the slope floor.
TAN_MIN = 0.001
TWI, FLAT = {}, {}
for t in PRODUCTS + ['SRTM_ROI']:
    sca = R(t, 'sca'); tanb = np.tan(np.radians(R(t, 'slope')))
    FLAT[t] = np.nanmean(np.where(np.isfinite(tanb), tanb < TAN_MIN, np.nan)) * 100
    TWI[t] = (np.log(sca / np.maximum(tanb, TAN_MIN))).astype('float32')
lo, hi = np.nanpercentile(np.concatenate([sample_values(TWI[t]) for t in PRODUCTS]), [2, 98])
compare_maps({t: TWI[t] for t in PRODUCTS}, 'Analysis 5a — Topographic Wetness Index (wetness POTENTIAL)',
             'TWI = ln(a / tan β)', 'YlGnBu', lo, hi, fname='A5a_TWI_maps.png', folder=OUT['05'])
def twi_stats(v):
    v = v[np.isfinite(v)]
    return {'mean': v.mean(), 'median': np.median(v), 'p5': np.percentile(v, 5), 'p95': np.percentile(v, 95)}
twi_tab = pd.DataFrame({LABEL[t]: {**twi_stats(TWI[t]), '% cells at flat-slope limit': FLAT[t]}
                        for t in PRODUCTS + ['SRTM_ROI']}).T
twi_tab.loc['LiDAR 1 m TWI averaged to 30 m'] = {**twi_stats(block_mean(TWI['LiDAR_1m'])), '% cells at flat-slope limit': np.nan}
show_table(twi_tab, 'A5_TWI_stats.csv', OUT['05'], 2)
r_src = diff_stats(TWI['SRTM_30m'], TWI['LiDAR_30m'])['r']
r_res = diff_stats(block_mean(TWI['LiDAR_1m']), TWI['LiDAR_30m'])['r']
print(f'Correlation of TWI on the 30 m grid: LiDAR 30 m vs SRTM r = {r_src:.2f} | '
      f'LiDAR 1 m (averaged) vs LiDAR 30 m r = {r_res:.2f}')

fig, ax = plt.subplots(figsize=(7.5, 4.4), constrained_layout=True)
for t in PRODUCTS:
    ax.hist(sample_values(TWI[t]), bins=80, range=(lo - 3, hi + 3), density=True, histtype='step', lw=2,
            color=COLOR[t], label=SHORT[t])
ax.set(xlabel='TWI', ylabel='Density', title='TWI distributions'); ax.legend()
fig.savefig(OUT['05'] / 'A5b_TWI_histograms.png', dpi=300); plt.show()
regional_map(TWI['SRTM_ROI'], 'Whole ROI — SRTM TWI', 'TWI', 'YlGnBu', lo, hi, fname='A5_regional_SRTM_TWI.png',
             folder=OUT['05'])
for t in PRODUCTS:
    export_raster(TWI[t], t, 'A5_TWI', OUT['05'])
add_summary('5 TWI', 'Mean TWI', '-', {t: np.nanmean(TWI[t]) for t in PRODUCTS})
add_summary('5 TWI', 'Cells at flat-slope limit', '%', {t: FLAT[t] for t in PRODUCTS})
add_summary('5 TWI', 'TWI correlation with LiDAR 30 m', 'r', {'LiDAR_1m': r_res, 'SRTM_30m': r_src})

**Interpretation guide.**
* TWI combines specific upslope contributing area and slope. Because both change with DEM resolution, values from different grid sizes may not have the same interpretation. Report distribution statistics and compare spatial patterns cautiously.
* In very flat areas the slope floor (`TAN_MIN`) can dominate the index. Inspect and report the fraction of affected cells rather than treating very high TWI as measured soil moisture.
* TWI inherits assumptions and artifacts from D8 routing and hydrological conditioning. A filled depression can have a routed contributing area even if runoff would actually pond there.

✏️ **Questions**
1. Do the highest TWI values coincide with mapped topographic depressions? Explain agreements and disagreements.
2. Which DEM provides defensible locations for a field soil-moisture sampling design, and how could you validate the index?


---
## Analysis 6 — Topographic depression mapping
**Principle.** A closed depression is a low area with no downhill outlet, so water collects there until it reaches the
**spill elevation** (the lowest point on its rim). Depressions are found on the **original, unfilled DEM** by filling every
depression to its spill level and subtracting the original surface:

> **potential ponding depth = filled surface − original surface**

Connected cells with depth > 0 form one depression. Depressions shallower than `MIN_DEPRESSION_DEPTH_M` or smaller than
`MIN_DEPRESSION_AREA_M2` are ignored (identical thresholds for all DEMs). Each depression is then classified:
* **uncertain (within DEM noise):** the maximum depth is smaller than the typical vertical uncertainty of the product
  (0.15 m for LiDAR, 2 m for SRTM), or it covers fewer than 3 cells;
* **possibly human-made (dugout/pit):** LiDAR only; deeper than 1.5 m but smaller than 0.3 ha (typical of farm dugouts);
* **edge:** touches the zoom-area edge, so its true spill point may lie outside;
* **likely natural:** all others.

**Question.** How many depressions does each DEM detect, how large are they, and which depressions are lost when the
terrain is coarsened?

**Processing notes (Analysis 6).** Detect depressions from **filled minus original** DEM, not from the already conditioned routing surface alone. Thresholds for depth and area help suppress tiny noise features, but thresholds can disproportionately affect 30 m products. The feature class labels are screening categories, not confirmed natural or artificial features.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 6: CLOSED DEPRESSION INVENTORY
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# Compute depth-to-spill from original terrain, not solely from the flow-conditioned DEM.
# The resulting inventory describes modelled surface depressions; it is NOT field-verified wetland mapping.
DEP = {}
for t in PRODUCTS + ['SRTM_ROI']:
    lab, depth, df = find_depressions(t)
    DEP[t] = {'lab': lab, 'depth': depth, 'df': df}
    print(f"{SHORT[t]:10s}: {len(df):5d} depressions | total area {df.area_m2.sum() / 1e4:8.1f} ha")
dmax = np.nanpercentile(DEP['LiDAR_1m']['depth'][DEP['LiDAR_1m']['depth'] > 0], 99) if (DEP['LiDAR_1m']['depth'] > 0).any() else 1
compare_maps({t: np.where(DEP[t]['depth'] > 0, DEP[t]['depth'], np.nan) for t in PRODUCTS},
             'Analysis 6a — Closed depressions: maximum potential ponding depth', 'Depth to spill level (m)',
             'viridis_r', 0, dmax, fname='A6a_depression_depth_maps.png', folder=OUT['06'], extend='max')

# inventory summary by class
inv_rows = []
for t in PRODUCTS:
    df = DEP[t]['df']
    for cl, g in df.groupby('class'):
        inv_rows.append({'product': SHORT[t], 'class': cl, 'count': len(g), 'area_ha': g.area_m2.sum() / 1e4,
                         'median_area_m2': g.area_m2.median(), 'mean_max_depth_m': g.max_depth_m.mean()})
inv_tab = pd.DataFrame(inv_rows).set_index(['product', 'class'])
show_table(inv_tab, 'A6_depression_counts_by_class.csv', OUT['06'], 2)

# overlap and depressions lost through coarsening (on the 1 m grid)
lab1 = DEP['LiDAR_1m']['lab']; df1 = DEP['LiDAR_1m']['df']
cover = {}
for t in ['LiDAR_30m', 'SRTM_30m']:
    m30 = upsample(DEP[t]['lab'] > 0)
    ov = np.bincount(lab1.ravel(), weights=m30.ravel(), minlength=len(df1) + 1)[1:] / np.maximum(df1.n_cells.values, 1)
    df1[f'overlap_with_{t}'] = ov
    cover[t] = ov
def size_class(a):
    return pd.cut(a, [0, 900, 4500, 1e12], labels=['< 0.09 ha (smaller than one 30 m cell)', '0.09-0.45 ha', '> 0.45 ha'])
lost_rows = []
for t in ['LiDAR_30m', 'SRTM_30m']:
    lost = cover[t] < 0.1
    for sc, g in df1.assign(lost=lost, size=size_class(df1.area_m2)).groupby('size', observed=False):
        lost_rows.append({'compared_with': SHORT[t], 'LiDAR 1 m size class': sc, 'n_LiDAR_1m': len(g),
                          'n_lost (<10% overlap)': int(g.lost.sum()),
                          '% lost': g.lost.mean() * 100 if len(g) else np.nan,
                          'storage_lost_m3': g.loc[g.lost, 'volume_m3'].sum()})
lost_tab = pd.DataFrame(lost_rows).set_index(['compared_with', 'LiDAR 1 m size class'])
show_table(lost_tab, 'A6_depressions_lost_by_coarsening.csv', OUT['06'], 1)
fp = {t: upsample(DEP[t]['lab'] > 0) if t != 'LiDAR_1m' else lab1 > 0 for t in PRODUCTS}
ov_tab = pd.DataFrame({SHORT[a]: {SHORT[b]: iou(fp[a], fp[b]) for b in PRODUCTS} for a in PRODUCTS})
print('Spatial overlap of depression footprints (IoU, 1 = identical):'); display(ov_tab.round(3))
ov_tab.to_csv(OUT['06'] / 'A6_depression_footprint_IoU.csv')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for t in PRODUCTS:
    df = DEP[t]['df']
    if len(df):
        ax[0].hist(np.log10(df.area_m2), bins=40, range=(2, 6), histtype='step', lw=2, color=COLOR[t], label=SHORT[t])
        ax[1].scatter(df.area_m2, df.max_depth_m, s=6, alpha=0.5, color=COLOR[t], label=SHORT[t])
ax[0].set(xlabel='log10(depression area, m²)', ylabel='Number of depressions', title='Depression size distribution')
ax[1].set(xscale='log', yscale='log', xlabel='Area (m²)', ylabel='Maximum depth (m)', title='Depression area vs depth')
for t in PRODUCTS:
    ax[1].axhline(VERTICAL_UNCERTAINTY_M[t], color=COLOR[t], ls=':', lw=1)
ax[0].legend(); ax[1].legend()
fig.savefig(OUT['06'] / 'A6b_depression_size_depth.png', dpi=300); plt.show()

dr = DEP['SRTM_ROI']['depth']
regional_map(np.where(dr > 0, dr, np.nan), 'Whole ROI — SRTM closed depressions', 'Depth (m)', 'viridis_r', 0,
             np.nanpercentile(dr[dr > 0], 99) if (dr > 0).any() else 1, fname='A6_regional_SRTM_depressions.png',
             folder=OUT['06'])
POLY = {}
for t in PRODUCTS + ['SRTM_ROI']:
    POLY[t] = depression_polygons(DEP[t]['lab'], DEP[t]['df'], t)
    POLY[t].to_file(OUT['06'] / 'A6_depressions.gpkg', layer=t, driver='GPKG')
    DEP[t]['df'].to_csv(OUT['06'] / f'A6_depression_inventory_{t}.csv', index=False)
    export_raster(DEP[t]['depth'], t, 'A6_ponding_depth_m', OUT['06'])
add_summary('6 Depressions', 'Number of depressions (all classes)', 'count', {t: len(DEP[t]['df']) for t in PRODUCTS})
add_summary('6 Depressions', 'Depressions classed likely natural', 'count',
            {t: (DEP[t]['df']['class'] == 'likely natural').sum() for t in PRODUCTS})
add_summary('6 Depressions', 'Depression area', 'ha', {t: DEP[t]['df'].area_m2.sum() / 1e4 for t in PRODUCTS})
add_summary('6 Depressions', 'LiDAR 1 m depressions lost', '%',
            {t: (cover[t] < 0.1).mean() * 100 if len(cover[t]) else np.nan for t in ['LiDAR_30m', 'SRTM_30m']})

**Interpretation guide.**
* Compare **observed** depression counts, sizes and depths in the exported inventories. LiDAR often captures smaller landforms, but the size distribution must be established from your own outputs.
* Aggregation may smooth rims, raise floors or merge nearby features; inspect how these changes influence depression identification and loss of small storage features.
* At 30 m, apparent SRTM depressions can reflect either landforms or elevation errors. Low overlap with LiDAR does **not alone prove** an individual depression is false: check vertical accuracy, alignment, field evidence and imagery.
* Some LiDAR depressions may be constructed (dugouts) or hydrologically connected by culverts absent from the DEM. The inventory's feature classes are screening aids, not validated labels.

✏️ **Questions**
1. Use the `A6_depressions.gpkg` inventory and the hillshade to identify one likely natural depression and one possibly human-made or artificial feature. What evidence supports your choices?
2. What share of native LiDAR depressions is not resolved by the two 30 m products? State your thresholds.


---
## Analysis 7 — Depression storage capacity
**Principle.** The maximum volume a depression can store before it spills is the sum of its ponding depths times the cell area:

> **storage (m³) = Σ (depth of each cell, m) × (cell area, m²)**, counting only cells inside the depression (up to its spill elevation).

Expressed per unit area of landscape (mm), storage tells you how much rainfall or snowmelt runoff the landscape can retain
before the depressions overflow.

**Question.** How much potential storage does each DEM report, and how much of the difference comes from small depressions
that coarse DEMs cannot resolve?

**Processing notes (Analysis 7).** Potential storage volume is **Σ(depth × cell area)** in m³. An equivalent depth in millimetres over the analysis footprint makes comparison with precipitation easier, but it does **not** predict actual ponding or infiltration. Study the small-depression contribution before comparing totals.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 7: POTENTIAL DEPRESSION STORAGE
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

# Storage capacity is a DEM-derived volume to the spill rim (m³), not a measured water volume.
# A larger total does not necessarily imply a more reliable DEM.
st_rows = {}
for t in PRODUCTS + ['SRTM_ROI']:
    df = DEP[t]['df']
    area_ha = np.isfinite(R(t, 'dem')).sum() * RES[t] ** 2 / 1e4
    ok = df['class'] == 'likely natural'
    st_rows[LABEL[t]] = {'n_depressions': len(df), 'total_storage_m3': df.volume_m3.sum(),
                         'storage_likely_natural_m3': df.loc[ok, 'volume_m3'].sum(),
                         'storage_uncertain_or_edge_m3': df.loc[~ok, 'volume_m3'].sum(),
                         'storage_depth_equivalent_mm': df.volume_m3.sum() / (area_ha * 1e4) * 1000,
                         'largest_depression_m3': df.volume_m3.max() if len(df) else 0,
                         'share_in_10_largest_%': (df.volume_m3.nlargest(10).sum() / df.volume_m3.sum() * 100)
                         if len(df) else np.nan}
st_tab = pd.DataFrame(st_rows).T
show_table(st_tab, 'A7_storage_summary.csv', OUT['07'], 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
x = np.arange(len(PRODUCTS))
nat = [st_rows[LABEL[t]]['storage_likely_natural_m3'] for t in PRODUCTS]
unc = [st_rows[LABEL[t]]['storage_uncertain_or_edge_m3'] for t in PRODUCTS]
ax[0].bar(x, nat, color=[COLOR[t] for t in PRODUCTS], label='likely natural / human-made excluded')
ax[0].bar(x, unc, bottom=nat, color=[COLOR[t] for t in PRODUCTS], alpha=0.35, hatch='//', label='uncertain or edge')
ax[0].set_xticks(x, [SHORT[t] for t in PRODUCTS]); ax[0].set(ylabel='Storage (m³)', title='Total potential depression storage')
ax[0].legend(fontsize=8)
for t in PRODUCTS:
    df = DEP[t]['df'].sort_values('area_m2')
    if len(df):
        ax[1].plot(df.area_m2, df.volume_m3.cumsum() / df.volume_m3.sum() * 100, color=COLOR[t], lw=2, label=SHORT[t])
ax[1].axvline(900, color='k', ls=':', lw=1); ax[1].text(950, 5, 'one 30 m cell', fontsize=8)
ax[1].set(xscale='log', xlabel='Depression area (m²)', ylabel='Cumulative share of storage (%)',
          title='Storage held by depressions up to a given size'); ax[1].legend()
fig.savefig(OUT['07'] / 'A7a_storage_comparison.png', dpi=300); plt.show()

def storage_overlay(ax, t):
    df = DEP[t]['df']
    if len(df):
        big = df.nlargest(15, 'volume_m3')
        ax.scatter(big.x, big.y, s=np.sqrt(big.volume_m3) * 2, facecolor='none', edgecolor='red', lw=1.2)
compare_maps({t: np.where(DEP[t]['depth'] > 0, DEP[t]['depth'], np.nan) for t in PRODUCTS},
             'Analysis 7b — Ponding depth; red circles = 15 largest storage volumes (circle size ∝ √volume)',
             'Depth (m)', 'viridis_r', 0, dmax, fname='A7b_storage_maps.png', folder=OUT['07'], overlay=storage_overlay,
             extend='max')
for t in PRODUCTS + ['SRTM_ROI']:
    DEP[t]['df'][['dep_id', 'area_m2', 'max_depth_m', 'mean_depth_m', 'volume_m3', 'spill_elev_m', 'x', 'y',
                  'class']].to_csv(OUT['07'] / f'A7_storage_by_depression_{t}.csv', index=False)
add_summary('7 Storage', 'Total depression storage', 'm³', {t: st_rows[LABEL[t]]['total_storage_m3'] for t in PRODUCTS})
add_summary('7 Storage', 'Storage as water depth over zoom area', 'mm',
            {t: st_rows[LABEL[t]]['storage_depth_equivalent_mm'] for t in PRODUCTS})

**Interpretation guide.**
* Compare the **measured model outputs** for potential storage rather than assuming a particular percentage difference. Coarser grids can remove small depressions, merge features, or alter rim and floor elevations; the net volume change is empirical and can differ across landforms.
* SRTM storage volumes are especially sensitive to vertical noise and the representation of shallow depressions. A large volume is not necessarily a more accurate volume.
* **Uncertainty:** vertical offset and local error; unrepresented culverts and berms; depression mergers; coarsening; acquisition date; and choice of spill elevation. For illustration, a 0.10 m depth change over 1 ha implies 1,000 m³ of storage difference.
* Storage expressed in mm of equivalent water depth over the analysis area is useful for comparing potential storage with precipitation or snowmelt, but **potential storage is not observed runoff**.

✏️ **Questions**
1. What share of native-LiDAR potential storage occurs in depressions smaller than one 30 m cell? Use the cumulative storage curve.
2. Which uncertainties seem most important for each DEM, and what additional data would reduce them?


---
## Analysis 8 — Surface-water connectivity (fill-and-spill network)
**Principle.** When a depression fills to its spill elevation, extra water overflows at the spill point and follows the
terrain downhill until it reaches the **next** depression (which then fills and may spill in turn) or leaves the area.
This **fill-and-spill** cascade is how prairie wetlands become hydrologically connected in wet years.

**Method.**
* Every depression is a **node**.
* From each depression's lowest point, the overflow path is traced cell by cell along the D8 directions of the conditioned
  DEM. Inside the filled depression these directions lead to the spill point; outside, they follow the terrain.
* The path ends at the first *other* depression it enters (an **edge**: A spills into B) or at the edge of the area
  (**outlet**: water leaves the zoom area).
* Each cell is also assigned to the first depression it drains to, which gives every depression its **local catchment**.

**Scenario definition.** All potential connections are shown for the **maximum-storage scenario**: every depression filled to
its spill elevation. These are **potential, topographic** connections. They do not prove that water was exchanged, and they
say nothing about when or how fast.

**Question.** How connected is the depression network according to each DEM, and how much land drains into connected
depression systems?

**Processing notes (Analysis 8).** The network is a *potential maximum-fill scenario*. Each mapped depression is a node and a possible downstream overflow is an edge. Connected components do not guarantee actual hydraulic exchange: rainfall, snowmelt, drainage works, frozen soils and evaporation are outside this DEM-only model.


In [ ]:
# ===========================================================================
# PRACTICAL 04 | ANALYSIS 8: FILL-AND-SPILL NETWORK
# Student guidance: run this cell after the previous cell(s); review figures/tables below.
# Scientific interpretation requires comparing common spatial extents and actual results.
# ===========================================================================

def depression_network(t):
    lab = DEP[t]['lab']; df = DEP[t]['df']; n = len(df)
    h, w = lab.shape
    nxt = downstream_index(R(t, 'pntr'))
    L = lab.ravel()
    if n == 0:
        return df.assign(target=[]), pd.DataFrame(columns=['from', 'to', 'path_length_m']), nx.DiGraph(), np.zeros(0)
    dem = np.where(np.isfinite(R(t, 'dem')), R(t, 'dem'), np.inf)
    rc = np.array(ndimage.minimum_position(dem, lab, df.dep_id.values))
    pos = rc[:, 0] * w + rc[:, 1]
    own = df.dep_id.values
    target = np.zeros(n, np.int64)           # >0: receiving depression; -1: leaves area; 0: unresolved
    active = np.ones(n, bool)
    every = max(1, int(round(GRID / RES[t])))
    paths = [[p] for p in pos]
    for step in range(4 * (h + w) * 4):
        a = np.flatnonzero(active)
        if a.size == 0:
            break
        nx_ = nxt[pos[a]]
        out = nx_ < 0
        target[a[out]] = -1; active[a[out]] = False
        a2 = a[~out]; pos[a2] = nx_[~out]
        lb = L[pos[a2]]
        hit = (lb > 0) & (lb != own[a2])
        target[a2[hit]] = lb[hit]; active[a2[hit]] = False
        if step % every == 0 or hit.any() or out.any():
            for i in a2:
                if step % every == 0 or not active[i]:
                    paths[i].append(pos[i])
    tr = TRANSFORM[t]
    def xy(p):
        r_, c_ = np.divmod(np.asarray(p), w)
        return np.column_stack([tr.c + (c_ + 0.5) * tr.a, tr.f + (r_ + 0.5) * tr.e])
    edges = []
    for i in range(n):
        if len(paths[i]) > 1:
            pts = xy(paths[i])
            geom = LineString(pts) if len(pts) > 1 else None
            edges.append({'from': int(own[i]), 'to': int(target[i]) if target[i] > 0 else -1,
                          'type': 'spills into depression' if target[i] > 0 else
                                  ('leaves zoom area' if target[i] == -1 else 'unresolved'),
                          'path_length_m': float(np.sum(np.hypot(*np.diff(pts, axis=0).T))), 'geometry': geom})
    E_ = gpd.GeoDataFrame(edges, geometry='geometry', crs=CRS) if edges else gpd.GeoDataFrame(columns=['from', 'to', 'geometry'])
    G = nx.DiGraph(); G.add_nodes_from(own.tolist())
    G.add_edges_from([(int(own[i]), int(target[i])) for i in range(n) if target[i] > 0])
    catch = first_downstream_label(nxt, lab)
    local_ha = np.bincount(catch, minlength=n + 1)[1:] * RES[t] ** 2 / 1e4
    return df.assign(target=target, local_catchment_ha=local_ha), E_, G, catch

# Connect potential overflow routes after depressions have filled to their spill elevations.
# Edges do not imply year-round streams or observed movement of water.
NET = {}
for t in PRODUCTS + ['SRTM_ROI']:
    t0 = time.time()
    nodes, edges, G, catch = depression_network(t)
    comps = [c for c in nx.weakly_connected_components(G) if len(c) > 1]
    comp_id = {nd: k + 1 for k, c in enumerate(sorted(comps, key=len, reverse=True)) for nd in c}
    nodes['network_id'] = nodes.dep_id.map(comp_id).fillna(0).astype(int)
    nodes['n_upstream_depressions'] = [len(nx.ancestors(G, d)) for d in nodes.dep_id] if len(nodes) else []
    NET[t] = {'nodes': nodes, 'edges': edges, 'G': G, 'comps': comps,
              'catch_area_ha': (catch > 0).sum() * RES[t] ** 2 / 1e4}
    print(f'{SHORT[t]:10s}: {G.number_of_nodes()} nodes, {G.number_of_edges()} spill connections ({time.time() - t0:.0f} s)')

net_rows = {}
for t in PRODUCTS + ['SRTM_ROI']:
    nd = NET[t]['nodes']; G = NET[t]['G']; comps = NET[t]['comps']
    area_ha = np.isfinite(R(t, 'dem')).sum() * RES[t] ** 2 / 1e4
    connected = set().union(*comps) if comps else set()
    net_rows[LABEL[t]] = {
        'depressions (nodes)': len(nd), 'spill connections (edges)': G.number_of_edges(),
        'connected depressions': len(connected),
        'isolated depressions (no link to another depression)': len(nd) - len(connected),
        'networks (≥ 2 depressions)': len(comps), 'largest network (depressions)': max((len(c) for c in comps), default=0),
        'longest spill cascade (steps)': nx.dag_longest_path_length(G) if G.number_of_edges() and nx.is_directed_acyclic_graph(G) else 0,
        'area draining to any depression (ha)': NET[t]['catch_area_ha'],
        '% of area draining to depressions': NET[t]['catch_area_ha'] / area_ha * 100,
        'connected drainage area (ha)': nd.loc[nd.network_id > 0, 'local_catchment_ha'].sum() if len(nd) else 0}
net_tab = pd.DataFrame(net_rows).T
show_table(net_tab, 'A8_connectivity_metrics.csv', OUT['08'], 1)

def net_overlay(ax, t):
    P_ = POLY[t]
    if len(P_):
        nid = NET[t]['nodes'].set_index('dep_id').network_id
        col = P_.dep_id.map(nid).fillna(0)
        P_[col == 0].plot(ax=ax, color='#9e9e9e', lw=0)
        if (col > 0).any():
            P_[col > 0].plot(ax=ax, column=col[col > 0] % 10, cmap='tab10', lw=0)
    e = NET[t]['edges']
    if len(e):
        e[e['type'] == 'spills into depression'].plot(ax=ax, color='#c2185b', lw=1.0)
        e[e['type'] == 'leaves zoom area'].plot(ax=ax, color='#c2185b', lw=0.5, ls=':', alpha=0.6)
compare_maps({t: np.full(SHAPE[t], np.nan) for t in PRODUCTS},
             'Analysis 8a — Fill-and-spill network (maximum-storage scenario; potential connections only)', '',
             fname='A8a_connectivity_maps.png', folder=OUT['08'], overlay=net_overlay,
             legend=[Patch(color='#9e9e9e', label='isolated depression'),
                     Patch(color=plt.cm.tab10(0), label='depression in a network (colour = network)'),
                     Line2D([], [], color='#c2185b', label='overflow path into next depression'),
                     Line2D([], [], color='#c2185b', ls=':', label='overflow leaving the area')])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
for t in PRODUCTS:
    sizes = sorted((len(c) for c in NET[t]['comps']), reverse=True)
    ax[0].plot(range(1, len(sizes) + 1), sizes, 'o-', color=COLOR[t], label=SHORT[t], ms=3)
    nd = NET[t]['nodes']
    if len(nd):
        ax[1].hist(np.log10(nd.local_catchment_ha.clip(lower=1e-4)), bins=40, histtype='step', lw=2, color=COLOR[t],
                   label=SHORT[t])
ax[0].set(xlabel='Network rank', ylabel='Depressions in network', yscale='log', title='Network sizes'); ax[0].legend()
ax[1].set(xlabel='log10(local catchment, ha)', ylabel='Depressions', title='Local catchment of each depression'); ax[1].legend()
fig.savefig(OUT['08'] / 'A8b_network_structure.png', dpi=300); plt.show()

regional_map(np.where(DEP['SRTM_ROI']['lab'] > 0, 1.0, np.nan), 'Whole ROI — SRTM fill-and-spill network', '',
             ListedColormap(['#1565c0']), 0, 1, fname='A8_regional_SRTM_connectivity.png', folder=OUT['08'],
             overlay=lambda ax: NET['SRTM_ROI']['edges'].plot(ax=ax, color='#c2185b', lw=0.6) if len(NET['SRTM_ROI']['edges']) else None,
             legend=[Patch(color='#1565c0', label='depression'), Line2D([], [], color='#c2185b', label='overflow path')])
for t in PRODUCTS + ['SRTM_ROI']:
    NET[t]['nodes'].to_csv(OUT['08'] / f'A8_nodes_{t}.csv', index=False)
    if len(NET[t]['edges']):
        NET[t]['edges'].to_file(OUT['08'] / 'A8_connectivity_edges.gpkg', layer=t, driver='GPKG')
add_summary('8 Connectivity', 'Spill connections between depressions', 'count',
            {t: NET[t]['G'].number_of_edges() for t in PRODUCTS})
add_summary('8 Connectivity', 'Connected depressions', 'count',
            {t: net_rows[LABEL[t]]['connected depressions'] for t in PRODUCTS})
add_summary('8 Connectivity', 'Connected drainage area', 'ha',
            {t: net_rows[LABEL[t]]['connected drainage area (ha)'] for t in PRODUCTS})
add_summary('8 Connectivity', 'Area draining to depressions', '%',
            {t: net_rows[LABEL[t]]['% of area draining to depressions'] for t in PRODUCTS})

**Interpretation guide.**
* Compare the number of modelled depression nodes, overflow edges, connected components and local contributing areas. Determine from the maps whether the networks follow natural swales, roadside ditches, or potentially erroneous DEM features.
* Differences can arise from changes in depression inventories, routing directions and spill thresholds. Similar network counts do **not** imply the same depressions are connected; inspect their locations and identities.
* Drainage to a modelled depression identifies **potential** internally stored flow under assumed conditions. It is not automatically equivalent to a measured, permanently non-contributing catchment.
* The displayed connections are a **potential maximum-storage fill-and-spill scenario**. Which overflow paths are active in a particular season depends on water supply, frozen soils, infiltration, evaporation, and drainage infrastructure; none are measured by a DEM alone.


**Analyses 1–8 complete.** Continue to the final comparison and reproducibility cells below. Your figures, tables and GIS outputs are stored in your own `Satellite_Practicals/Practical_04/<student_folder>/` directory.


---
## 9. Final comparison and reproducibility check
The eight analyses have already produced GeoTIFF layers, PNG figures, vector network products and statistical tables in your Drive. The next cell generates a **run configuration**, a **combined comparison summary** and a **file manifest**. These let you verify which files actually exist, without claiming that failed or interrupted analyses succeeded.

Before submission, check that the **analyses 01–08 folders** contain outputs, and make sure every discussion compares the same footprint, with units and physical interpretation. If `MODE='QUICK'`, do **not** claim to have analyzed the full LiDAR coverage.


In [ ]:
# ===== Final inventory: save metadata and summarize files actually produced =====
from datetime import datetime, timezone

CONFIG = {
    'course': 'Satellite Remote Sensing Practicals / Practical 04',
    'case_study': 'Elm Creek, Manitoba',
    'student_folder': STUDENT_FOLDER_SAFE,
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'earth_engine_project': GEE_PROJECT_ID,
    'mode': MODE,
    'region_crs': CRS,
    'comparison_grid_m': GRID,
    'quick_box_utm': QUICK_BOX,
    'processing_window_utm': ZOOM_BOUNDS,
    'condition_routing': CONDITIONING,
    'stream_threshold_ha': STREAM_THRESHOLD_HA,
    'minimum_depression_depth_m': MIN_DEPRESSION_DEPTH_M,
    'minimum_depression_area_m2': MIN_DEPRESSION_AREA_M2,
    'watershed_outlet_utm': [OUTLET_X, OUTLET_Y],
    'outlet_snap_distance_m': SNAP_DISTANCE_M,
    'source_retrieval': 'Automatic instructor folders; no student source paths',
    'source_roi_folder_url': ROI_FOLDER_URL,
    'source_data_folder_url': LIDAR_FOLDER_URL,
    'lidar_archive_paths': {name: str(path) for name, path in LIDAR_ZIPS.items()},
    'lidar_roi_source': str(roi_shp),
    'zoom_kmz_source': str(kmz_path),
    'srtm_earth_engine_collection': 'USGS/SRTMGL1_003',
    'notes': 'Terrain-derived hydrological potential, not measured discharge or water storage.'
}
(OUT_ROOT / 'run_configuration.json').write_text(json.dumps(CONFIG, indent=2, default=str))

if SUMMARY:
    summary_df = pd.DataFrame(SUMMARY)
    summary_df.to_csv(OUT_ROOT / 'eight_analyses_comparison_summary.csv', index=False)
    display(summary_df.head(15))
else:
    print('No summary metrics have been recorded. Confirm the eight analysis cells ran.')

inventory = []
for folder in OUT.values():
    for p in folder.rglob('*'):
        if p.is_file():
            inventory.append({'analysis_folder': folder.name,
                              'relative_path': str(p.relative_to(OUT_ROOT)),
                              'file_type': p.suffix.lower(),
                              'size_MB': round(p.stat().st_size / 1e6, 3)})
manifest_df = pd.DataFrame(inventory, columns=['analysis_folder','relative_path','file_type','size_MB'])
manifest_df.to_csv(OUT_ROOT / 'output_manifest.csv', index=False)
print(f'Individual result folder: {OUT_ROOT}')
print(f'Inventory: {len(manifest_df)} files, excluding cached data and final metadata.')
print('Run configuration and manifest have been saved.')


## 10. Scientific interpretation and student deliverables
### Suggested practical submission
1. **Dataset quality and setup:** Identify the study area, DEM source/resolution, output coordinate system, and whether QUICK or FULL mode was used.
2. **Four representative comparison figures:** Include elevation/slope, drainage accumulation, depression depth, and fill-and-spill connectivity, using scientific figure captions and units.
3. **Four supplementary results:** A watershed map, D8 flow-direction plot, TWI map, and storage summary.
4. **A comparison table** with at least six quantitative metrics drawn from the notebook (e.g. elevation RMSE, drainage area, watershed IoU, TWI summary, depression count, potential storage and network edges).
5. **Scientific interpretation** (recommended 600–900 words, unless your instructor specifies otherwise): distinguish differences caused by spatial resolution from differences caused by DEM source; discuss uncertainty in extremely flat prairie terrain.
6. **Reproducibility:** Include `run_configuration.json`, `output_manifest.csv`, and the required maps/tables already saved in your Drive. Submit the files requested through your course's official assignment page; **saving to Drive does not automatically submit the assignment**.

### Questions for independent interpretation
- At matched 30 m resolution, does SRTM produce systematically different elevations and slopes from the aggregated LiDAR DEM? How large is the difference and what other explanations exist besides measurement accuracy?
- Do the different products map the same drainage lines, watershed boundaries, closed depressions and spill connections? Use at least one **quantitative** measure as well as a map.
- How does a common **10 ha drainage threshold** translate to upstream cell counts at 1 m versus 30 m? Why would a common number of cells be misleading?
- Which outputs could help identify a field where excess water might pond? Which additional data (e.g. soils, precipitation, culverts, field surveys) would be needed before making a management decision?
- Why should a *filled* routing DEM never be used as the sole source for estimating the original landscape's storage capacity?

### Important limitations
**A DEM is a static topographic representation, not a physically complete hydrological simulation.** D8 simplifies flow to one downstream neighbour; conditioning can create artificial spill paths; drainage infrastructure may not be represented; and vertical-error differences can strongly affect shallow depressions. Student interpretations must recognize these limits.

### References and technical documentation
- [Google Earth Engine SRTM 30 m dataset documentation](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003) — product metadata, resolution and acquisition period.
- [WhiteboxTools project](https://www.whiteboxgeo.com/whitebox-geospatial-analysis-tools/) — tools for DEM conditioning, routing and hydrological terrain analysis.
- [Google Colab guidance](https://research.google.com/colaboratory/faq.html) — runtime, storage and Drive mounting behavior.
- O'Callaghan, J. F. & Mark, D. M. (1984). The extraction of drainage networks from digital elevation data. *Computer Vision, Graphics, and Image Processing*, 28, 323–344.
- Beven, K. J. & Kirkby, M. J. (1979). A physically based, variable contributing area model of basin hydrology. *Hydrological Sciences Bulletin*, 24(1), 43–69.
- Farr, T. G. et al. (2007). The Shuttle Radar Topography Mission. *Reviews of Geophysics*, 45, RG2004.

**End of Practical 04.** Preserve your original scientific observations in your own report. The default input polygons, LiDAR files and hydrological algorithms are adapted from the instructor-supplied SOIL 7320 DEM hydrology notebook.
